import pandas as pd

## Load Datasets

I will now load the `train.csv.zip` and `test.csv.zip` files into pandas DataFrames. The `compression='zip'` argument ensures that pandas can directly read from the zipped files.

In [8]:
try:
    df_train = pd.read_csv('/content/train.csv.zip', compression='zip')
    print("Train data loaded successfully.")
    display(df_train.head())
except FileNotFoundError:
    print("Error: train.csv.zip not found. Please ensure the file is in /content/.")
except Exception as e:
    print(f"An error occurred while loading train data: {e}")

Error: train.csv.zip not found. Please ensure the file is in /content/.


In [9]:
try:
    df_test = pd.read_csv('/content/test.csv.zip', compression='zip')
    print("\nTest data loaded successfully.")
    display(df_test.head())
except FileNotFoundError:
    print("Error: test.csv.zip not found. Please ensure the file is in /content/.")
except Exception as e:
    print(f"An error occurred while loading test data: {e}")

Error: test.csv.zip not found. Please ensure the file is in /content/.


## Exploratory Data Analysis (EDA) and Preprocessing

Before building the model, it's crucial to understand the data. I'll start by checking the general information of both `df_train` and `df_test` to identify data types, non-null counts, and memory usage. This will help us spot inconsistencies or missing values that need to be addressed.

In [10]:
print("\n--- Training Data Info ---")
df_train.info()
print("\n--- Test Data Info ---")
df_test.info()


--- Training Data Info ---


NameError: name 'df_train' is not defined

Next, I'll examine the descriptive statistics for numerical columns to understand their distribution, central tendency, and spread. This can highlight potential outliers or scaling issues.

In [11]:
print("\n--- Training Data Description ---")
display(df_train.describe(include='all'))
print("\n--- Test Data Description ---")
display(df_test.describe(include='all'))


--- Training Data Description ---


NameError: name 'df_train' is not defined

The `DtypeWarning` during loading suggested mixed types in column 26 of the training data. Let's inspect the `Credit_Score` column (which appears to be column 26 based on the dataframe's structure) and `Age` column for unique values and non-numeric entries, as these are critical features and potential sources of issues.

In [12]:
print("\nUnique values in 'Credit_Score' (train):", df_train['Credit_Score'].unique())
print("\nUnique values in 'Age' (train):", df_train['Age'].unique())
print("\nUnique values in 'Age' (test):", df_test['Age'].unique())

NameError: name 'df_train' is not defined

## Data Cleaning and Type Conversion

Based on the EDA, several columns require cleaning and type conversion. I'll address these issues systematically for both `df_train` and `df_test`.

In [13]:
def clean_and_convert_data(df):
    # Convert 'Credit_Score' to numerical using Label Encoding (Poor: 0, Standard: 1, Good: 2)
    credit_score_mapping = {'Poor': 0, 'Standard': 1, 'Good': 2}
    if 'Credit_Score' in df.columns:
        df['Credit_Score'] = df['Credit_Score'].map(credit_score_mapping)

    # Clean and convert 'Age' column
    def clean_age(age_val):
        if isinstance(age_val, str):
            age_val = age_val.replace('_', '').replace('+', '')
            if age_val.isdigit():
                age = int(age_val)
                if 0 < age < 100: # Assuming realistic age is between 0 and 100
                    return age
                elif age < 0: # Handle negative ages by taking absolute value or NaN
                    return abs(age) # Or np.nan depending on desired imputation strategy
                else: # Handle unrealistic large ages
                    return np.nan
            else:
                return np.nan # Not a digit after cleaning
        return age_val # Return as is if already numeric (and hopefully valid)

    df['Age'] = df['Age'].apply(clean_age)
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    # Replace unrealistic large 'Age' values (e.g., >100) with NaN for later imputation
    df.loc[df['Age'] > 100, 'Age'] = np.nan

    # Clean and convert financial object columns to numeric
    financial_cols = [
        'Annual_Income', 'Num_of_Loan', 'Outstanding_Debt',
        'Amount_invested_monthly', 'Monthly_Balance'
    ]
    for col in financial_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(r'[^\d.]', '', regex=True) # Remove non-numeric except dot
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Clean and convert 'Credit_History_Age' to years
    def convert_credit_history_age(age_str):
        if isinstance(age_str, str):
            parts = age_str.replace('Years', '').replace('Months', '').replace('and', '').split()
            years = 0
            months = 0
            if len(parts) >= 2:
                try:
                    years = int(parts[0])
                except ValueError:
                    pass
                try:
                    months = int(parts[1])
                except ValueError:
                    pass
            return years + months / 12
        return np.nan

    if 'Credit_History_Age' in df.columns:
        df['Credit_History_Age'] = df['Credit_History_Age'].apply(convert_credit_history_age)

    # Handle problematic '-1' in 'Num_Bank_Accounts' and 'Num_of_Loan' by replacing with NaN
    for col in ['Num_Bank_Accounts', 'Num_of_Loan']:
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan

    return df

import numpy as np

df_train = clean_and_convert_data(df_train)
df_test = clean_and_convert_data(df_test)

print("Data cleaning and conversion complete.")

print("\n--- Training Data Info After Cleaning ---")
df_train.info()
print("\n--- Test Data Info After Cleaning ---")
df_test.info()

NameError: name 'df_train' is not defined

Now, let's re-examine the descriptive statistics for the cleaned numerical columns and unique values for critical columns like `Age` and `Monthly_Balance` to ensure the cleaning was effective.

In [ ]:
print("\n--- Training Data Description After Cleaning ---")
display(df_train.describe(include='all'))
print("\n--- Test Data Description After Cleaning ---")
display(df_test.describe(include='all'))

print("\nUnique values in 'Age' (train) after cleaning:", df_train['Age'].unique())
print("\nUnique values in 'Age' (test) after cleaning:", df_test['Age'].unique())

print("\nUnique values in 'Monthly_Balance' (train) after cleaning:", df_train['Monthly_Balance'].unique())
print("\nUnique values in 'Monthly_Balance' (test) after cleaning:", df_test['Monthly_Balance'].unique())

## Handling Missing Values

Now that data types are largely corrected, I will address the missing values. I'll use different strategies based on the column type and nature:
- **Numerical columns**: Impute with median (less sensitive to outliers than mean).
- **Categorical columns**: Impute with mode (most frequent category).
- **'Credit_History_Age'**: Impute with median.

First, let's identify the columns with missing values and their types to determine the best imputation strategy.

In [ ]:
print("\n--- Missing values in Training Data ---")
print(df_train.isnull().sum()[df_train.isnull().sum() > 0])

print("\n--- Missing values in Test Data ---")
print(df_test.isnull().sum()[df_test.isnull().sum() > 0])

# Imputation Strategy
# Numerical columns: Age, Monthly_Inhand_Salary, Num_Bank_Accounts, Num_of_Loan, Num_Credit_Inquiries, Credit_History_Age, Amount_invested_monthly, Monthly_Balance
# Categorical columns: Name, Type_of_Loan, Num_of_Delayed_Payment

# Impute numerical columns with median
numeric_cols_to_impute = [
    'Age', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_of_Loan',
    'Num_Credit_Inquiries', 'Credit_History_Age', 'Amount_invested_monthly', 'Monthly_Balance'
]

for col in numeric_cols_to_impute:
    if col in df_train.columns:
        median_val_train = df_train[col].median()
        df_train[col].fillna(median_val_train, inplace=True)
    if col in df_test.columns:
        median_val_test = df_test[col].median() # Use test median for test set
        df_test[col].fillna(median_val_test, inplace=True)

# Impute categorical columns with mode
categorical_cols_to_impute = [
    'Name', 'Type_of_Loan', 'Num_of_Delayed_Payment'
]

for col in categorical_cols_to_impute:
    if col in df_train.columns:
        mode_val_train = df_train[col].mode()[0]
        df_train[col].fillna(mode_val_train, inplace=True)
    if col in df_test.columns:
        mode_val_test = df_test[col].mode()[0] # Use test mode for test set
        df_test[col].fillna(mode_val_test, inplace=True)

print("\n--- Missing values after imputation in Training Data ---")
print(df_train.isnull().sum().sum()) # Should be 0

print("\n--- Missing values after imputation in Test Data ---")
print(df_test.isnull().sum().sum()) # Should be 0

print("Missing value imputation complete.")

## Categorical Feature Encoding

With missing values handled, the next step is to encode the remaining categorical 'object' columns into numerical representations suitable for machine learning models. I will use one-hot encoding for nominal categories and Label Encoding for ordinal categories where appropriate.

Let's first identify all 'object' type columns that are not IDs or the target variable.

In [14]:
import numpy as np

# Identify categorical columns (excluding IDs and target)
categorical_cols = [col for col in df_train.select_dtypes(include='object').columns
                    if col not in ['ID', 'Customer_ID', 'SSN']]

print("Categorical columns to encode:", categorical_cols)

# Apply One-Hot Encoding to the identified categorical columns
df_train = pd.get_dummies(df_train, columns=categorical_cols, drop_first=True)
df_test = pd.get_dummies(df_test, columns=categorical_cols, drop_first=True)

print("\nShape of training data after one-hot encoding:", df_train.shape)
print("Shape of test data after one-hot encoding:", df_test.shape)

# Align columns - crucial for consistent features between train and test sets
train_cols = set(df_train.columns)
test_cols = set(df_test.columns)

missing_in_test = list(train_cols - test_cols)
for col in missing_in_test:
    if col != 'Credit_Score': # Don't add Credit_Score to test set
        df_test[col] = 0

missing_in_train = list(test_cols - train_cols)
for col in missing_in_train:
    df_train[col] = 0

# Ensure the order of columns is the same
df_test = df_test[df_train.drop(columns=['Credit_Score']).columns]

print("\nData preparation (missing values and categorical encoding) complete.")

print("\n--- Final Training Data Info ---")
df_train.info()
print("\n--- Final Test Data Info ---")
df_test.info()

print("\n--- Final Training Data Head ---")
display(df_train.head())
print("\n--- Final Test Data Head ---")
display(df_test.head())

NameError: name 'df_train' is not defined

## Model Preparation and Training (Logistic Regression Baseline)

With the data now cleaned and preprocessed, I'll prepare it for modeling by separating the features (X) from the target variable (y). I'll then train a Logistic Regression model as a baseline for the credit scoring prediction task.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Define features (X) and target (y) for the training set
# Drop ID, Customer_ID, and SSN as they are identifiers and not features.
# The 'Name' column was one-hot encoded, so we don't need to drop it here directly if its components are already created.
# The 'Credit_Score' column is our target variable.

# Drop non-feature columns from train and test sets for consistency
non_feature_cols = ['ID', 'Customer_ID', 'SSN']
X_train = df_train.drop(columns=['Credit_Score'] + [col for col in non_feature_cols if col in df_train.columns])
y_train = df_train['Credit_Score']

X_test = df_test.drop(columns=[col for col in non_feature_cols if col in df_test.columns])

# Align columns after one-hot encoding might lead to different columns in train and test.
# Ensure X_test has the same columns as X_train, filling missing with 0 and dropping extra.
# This step is crucial if some categories appeared in train but not test or vice-versa.
missing_cols = set(X_train.columns) - set(X_test.columns)
for c in missing_cols:
    X_test[c] = 0
# Ensure the order of columns in the test set is the same as in the training set
X_test = X_test[X_train.columns]

print("Shape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)

# Initialize and train a Logistic Regression model
# Using solver='liblinear' for smaller datasets or for L1/L2 regularization
# Increased max_iter for convergence on potentially complex data
model = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
model.fit(X_train, y_train)

print("\nLogistic Regression model trained successfully.")

## Model Evaluation

Now I will evaluate the performance of the trained Logistic Regression model on the training data using the requested metrics: Precision, Recall, F1-Score, and ROC-AUC. Since we don't have a separate validation set yet, I'll evaluate on the training data, but it's important to note that this will give an optimistic view of performance.

In [ ]:
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Re-define and train the model to ensure it's available
# This part is moved here to address the NameError if the previous cell's state was lost
# Define features (X) and target (y) for the training set (assuming X_train, y_train are already defined from previous steps)
non_feature_cols = ['ID', 'Customer_ID', 'SSN']
X_train = df_train.drop(columns=['Credit_Score'] + [col for col in non_feature_cols if col in df_train.columns])
y_train = df_train['Credit_Score']

model = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
model.fit(X_train, y_train)
print("Logistic Regression model re-trained successfully within this cell.")

# Predict on the training data
y_pred_train = model.predict(X_train)
y_pred_proba_train = model.predict_proba(X_train)

# Evaluate the model on the training data
print("\n--- Model Evaluation on Training Data ---")
print(f"Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Precision (macro): {precision_score(y_train, y_pred_train, average='macro'):.4f}")
print(f"Recall (macro): {recall_score(y_train, y_pred_train, average='macro'):.4f}")
print(f"F1-Score (macro): {f1_score(y_train, y_pred_train, average='macro'):.4f}")

# It's better to use classification_report for detailed metrics per class
print("\nClassification Report (Training Data):")
print(classification_report(y_train, y_pred_train))

# For ROC-AUC in multi-class, we often look at it per class or averaged.
# Given 'Credit_Score' is ordinal ('Poor', 'Standard', 'Good'), we can use 'ovr' strategy.
roc_auc_ovr = roc_auc_score(y_train, y_pred_proba_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (macro, One-vs-Rest): {roc_auc_ovr:.4f}")

## Training a Random Forest Classifier

Now, let's train another classification model, a Random Forest Classifier, and compare its performance to the Logistic Regression baseline. Random Forest can capture more complex relationships in the data.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize and train a Random Forest model
# Using n_estimators (number of trees), random_state for reproducibility
# and class_weight to handle potential class imbalance (if any, though not explicitly checked yet)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

print("Random Forest Classifier trained successfully.")

## Random Forest Model Evaluation

Now, let's evaluate the performance of the Random Forest model on the training data using the same metrics as before.

In [15]:
from sklearn.metrics import classification_report

# Predict on the training data using Random Forest
y_pred_rf_train = rf_model.predict(X_train)
y_pred_proba_rf_train = rf_model.predict_proba(X_train)

# Evaluate the Random Forest model on the training data
print("\n--- Random Forest Model Evaluation on Training Data ---")
print(f"Accuracy: {accuracy_score(y_train, y_pred_rf_train):.4f}")
print(f"Precision (macro): {precision_score(y_train, y_pred_rf_train, average='macro'):.4f}")
print(f"Recall (macro): {recall_score(y_train, y_pred_rf_train, average='macro'):.4f}")
print(f"F1-Score (macro): {f1_score(y_train, y_pred_rf_train, average='macro'):.4f}")

print("\nClassification Report (Random Forest, Training Data):")
print(classification_report(y_train, y_pred_rf_train))

roc_auc_rf_ovr = roc_auc_score(y_train, y_pred_proba_rf_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (Random Forest, macro, One-vs-Rest): {roc_auc_rf_ovr:.4f}")

NameError: name 'rf_model' is not defined

## Training a Decision Tree Classifier

To continue exploring the classification algorithms, I will now train a Decision Tree Classifier. Decision Trees are interpretable models that can provide insights into feature importance.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Initialize and train a Decision Tree model
# Using random_state for reproducibility and class_weight='balanced'
# Limiting max_depth to prevent immediate overfitting, but this can be tuned later.
dt_model = DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10)
dt_model.fit(X_train, y_train)

print("Decision Tree Classifier trained successfully.")

## Decision Tree Model Evaluation

Now, let's evaluate the performance of the Decision Tree model on the training data using the standard classification metrics.

In [ ]:
from sklearn.metrics import classification_report

# Predict on the training data using Decision Tree
y_pred_dt_train = dt_model.predict(X_train)
y_pred_proba_dt_train = dt_model.predict_proba(X_train)

# Evaluate the Decision Tree model on the training data
print("\n--- Decision Tree Model Evaluation on Training Data ---")
print(f"Accuracy: {accuracy_score(y_train, y_pred_dt_train):.4f}")
print(f"Precision (macro): {precision_score(y_train, y_pred_dt_train, average='macro'):.4f}")
print(f"Recall (macro): {recall_score(y_train, y_pred_dt_train, average='macro'):.4f}")
print(f"F1-Score (macro): {f1_score(y_train, y_pred_dt_train, average='macro'):.4f}")

print("\nClassification Report (Decision Tree, Training Data):")
print(classification_report(y_train, y_pred_dt_train))

roc_auc_dt_ovr = roc_auc_score(y_train, y_pred_proba_dt_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (Decision Tree, macro, One-vs-Rest): {roc_auc_dt_ovr:.4f}")

## Feature Engineering

Now, let's create some new features that might be beneficial for predicting creditworthiness. Based on typical financial analysis, these could include ratios, interactions, or transformations of existing data.

Potential features to engineer:
- **Debt-to-Income Ratio**: Outstanding_Debt / Annual_Income
- **Payment to Income Ratio**: Total_EMI_per_month / Monthly_Inhand_Salary
- **Credit Utilization Status**: A categorical feature based on Credit_Utilization_Ratio (e.g., 'Low', 'Medium', 'High')
- **Ratio of Monthly Inhand Salary to Annual Income**: (Monthly_Inhand_Salary * 12) / Annual_Income
- **Credit History Length (numeric)**: Already converted 'Credit_History_Age' to years, but could explore bins.
- **Balance to Income Ratio**: Monthly_Balance / Monthly_Inhand_Salary

In [16]:
def create_financial_features(df):
    df_copy = df.copy()

    # Convert columns to numeric, coercing errors to NaN
    numeric_cols = [
        'Annual_Income', 'Monthly_Inhand_Salary', 'Outstanding_Debt',
        'Total_EMI_per_month', 'Monthly_Balance', 'Credit_Utilization_Ratio'
    ]
    for col in numeric_cols:
        if col in df_copy.columns:
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')
            # Impute NaNs with median after conversion for newly created NaNs
            if df_copy[col].isnull().any():
                df_copy[col].fillna(df_copy[col].median(), inplace=True)

    # 1. Debt-to-Income Ratio
    # Handle potential division by zero by replacing 0 annual income with a small epsilon or median
    # For simplicity, let's add a small constant to avoid division by zero
    df_copy['Annual_Income_adj'] = df_copy['Annual_Income'].replace(0, df_copy['Annual_Income'].median())
    df_copy['Debt_to_Income_Ratio'] = df_copy['Outstanding_Debt'] / df_copy['Annual_Income_adj']
    df_copy.drop('Annual_Income_adj', axis=1, inplace=True)

    # 2. Payment to Income Ratio
    df_copy['Monthly_Inhand_Salary_adj'] = df_copy['Monthly_Inhand_Salary'].replace(0, df_copy['Monthly_Inhand_Salary'].median())
    df_copy['Payment_to_Income_Ratio'] = df_copy['Total_EMI_per_month'] / df_copy['Monthly_Inhand_Salary_adj']
    df_copy.drop('Monthly_Inhand_Salary_adj', axis=1, inplace=True)

    # 3. Credit Utilization Status (Categorical, then one-hot encode later if needed)
    # For now, let's keep it numeric or just as ratio.
    # We already have 'Credit_Utilization_Ratio', which is a good direct feature.

    # 4. Ratio of Monthly Inhand Salary to Annual Income (as a consistency check)
    df_copy['Annual_Income_adj'] = df_copy['Annual_Income'].replace(0, df_copy['Annual_Income'].median())
    df_copy['Monthly_Income_Annual_Ratio'] = (df_copy['Monthly_Inhand_Salary'] * 12) / df_copy['Annual_Income_adj']
    df_copy.drop('Annual_Income_adj', axis=1, inplace=True)

    # 5. Balance to Income Ratio
    df_copy['Monthly_Inhand_Salary_adj'] = df_copy['Monthly_Inhand_Salary'].replace(0, df_copy['Monthly_Inhand_Salary'].median())
    df_copy['Balance_to_Income_Ratio'] = df_copy['Monthly_Balance'] / df_copy['Monthly_Inhand_Salary_adj']
    df_copy.drop('Monthly_Inhand_Salary_adj', axis=1, inplace=True)

    # Fill any NaNs generated during feature creation (e.g., from division by zero or original NaNs)
    for col in ['Debt_to_Income_Ratio', 'Payment_to_Income_Ratio', 'Monthly_Income_Annual_Ratio', 'Balance_to_Income_Ratio']:
        if col in df_copy.columns:
            df_copy[col].fillna(df_copy[col].median(), inplace=True)

    return df_copy

# Apply feature engineering to both training and test datasets
df_train_fe = create_financial_features(df_train)
df_test_fe = create_financial_features(df_test)

print("Feature engineering complete. New features added to df_train_fe and df_test_fe.")

print("\n--- Training Data Info After Feature Engineering ---")
df_train_fe.info()
print("\n--- Test Data Info After Feature Engineering ---")
df_test_fe.info()

NameError: name 'df_train' is not defined

## Re-training and Re-evaluating Models with New Features

Now that we have new engineered features, let's prepare the data again (separate X and y, and align columns) and then re-train and evaluate our baseline Logistic Regression model to see if the new features have improved performance.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

# Define features (X) and target (y) for the training set with new features
non_feature_cols = ['ID', 'Customer_ID', 'SSN']
X_train_fe = df_train_fe.drop(columns=['Credit_Score'] + [col for col in non_feature_cols if col in df_train_fe.columns])
y_train_fe = df_train_fe['Credit_Score']

X_test_fe = df_test_fe.drop(columns=[col for col in non_feature_cols if col in df_test_fe.columns])

# Align columns after feature engineering
train_fe_cols = set(X_train_fe.columns)
test_fe_cols = set(X_test_fe.columns)

missing_in_test_fe = list(train_fe_cols - test_fe_cols)
for col in missing_in_test_fe:
    X_test_fe[col] = 0

missing_in_train_fe = list(test_fe_cols - train_fe_cols)
for col in missing_in_train_fe:
    X_train_fe[col] = 0

# Ensure the order of columns is the same
X_test_fe = X_test_fe[X_train_fe.columns]

print("Shape of X_train_fe:", X_train_fe.shape)
print("Shape of y_train_fe:", y_train_fe.shape)
print("Shape of X_test_fe:", X_test_fe.shape)

# Re-initialize and train a Logistic Regression model with new features
lr_model_fe = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
lr_model_fe.fit(X_train_fe, y_train_fe)

print("\nLogistic Regression model re-trained with new features successfully.")

# Predict and evaluate on the training data
y_pred_lr_fe_train = lr_model_fe.predict(X_train_fe)
y_pred_proba_lr_fe_train = lr_model_fe.predict_proba(X_train_fe)

print("\n--- Logistic Regression Model Evaluation (with FE) on Training Data ---")
print(f"Accuracy: {accuracy_score(y_train_fe, y_pred_lr_fe_train):.4f}")
print(f"Precision (macro): {precision_score(y_train_fe, y_pred_lr_fe_train, average='macro'):.4f}")
print(f"Recall (macro): {recall_score(y_train_fe, y_pred_lr_fe_train, average='macro'):.4f}")
print(f"F1-Score (macro): {f1_score(y_train_fe, y_pred_lr_fe_train, average='macro'):.4f}")
print("\nClassification Report (Logistic Regression, with FE, Training Data):")
print(classification_report(y_train_fe, y_pred_lr_fe_train))
roc_auc_lr_fe_ovr = roc_auc_score(y_train_fe, y_pred_proba_lr_fe_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (Logistic Regression, with FE, macro, One-vs-Rest): {roc_auc_lr_fe_ovr:.4f}")

## Hyperparameter Tuning for Random Forest Classifier

Given that the Random Forest Classifier showed the best cross-validated performance, let's now optimize its hyperparameters using `GridSearchCV`. This will help us find the best combination of parameters for improved generalization and performance.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define the parameter grid to search
param_grid = {
    'n_estimators': [100, 200, 300], # Number of trees in the forest
    'max_features': ['sqrt', 'log2'], # Number of features to consider at each split
    'max_depth': [10, 20, 30], # Maximum depth of the tree
    'min_samples_split': [2, 5], # Minimum number of samples required to split a node
    'min_samples_leaf': [1, 2], # Minimum number of samples required at a leaf node
    'class_weight': ['balanced'] # Keep 'balanced' as determined before
}

# Initialize the Random Forest Classifier
rf_base = RandomForestClassifier(random_state=42)

# Initialize GridSearchCV
# We'll use the StratifiedKFold (skf) from the previous cross-validation step
# scoring='roc_auc_ovr' to optimize for ROC-AUC
# n_jobs=-1 to use all available cores
# verbose=2 for detailed output
grid_search = GridSearchCV(estimator=rf_base, param_grid=param_grid,
                           cv=skf, n_jobs=-1, verbose=2, scoring='roc_auc_ovr')

print("Starting GridSearchCV for Random Forest. This may take a while...")

# Fit GridSearchCV to the data
# Assuming X_train_fe and y_train_fe are available from previous steps
grid_search.fit(X_train_fe, y_train_fe)

print("GridSearchCV complete.")

# Get the best parameters and best score
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"\nBest parameters found: {best_params}")
print(f"Best ROC-AUC score found during tuning: {best_score:.4f}")

# Train the Random Forest model with the best parameters
best_rf_model = RandomForestClassifier(**best_params, random_state=42)
best_rf_model.fit(X_train_fe, y_train_fe)

print("Random Forest model trained with best parameters.")

## Evaluating the Tuned Random Forest Model

Now that we have tuned the Random Forest Classifier and trained it with the best parameters, let's evaluate its performance on the training data one last time to see the impact of tuning. This will be followed by making predictions on the test set.

In [ ]:
from sklearn.metrics import classification_report

# Predict on the training data using the best Random Forest model
y_pred_best_rf_train = best_rf_model.predict(X_train_fe)
y_pred_proba_best_rf_train = best_rf_model.predict_proba(X_train_fe)

# Evaluate the best Random Forest model on the training data
print("\n--- Tuned Random Forest Model Evaluation on Training Data ---")
print(f"Accuracy: {accuracy_score(y_train_fe, y_pred_best_rf_train):.4f}")
print(f"Precision (macro): {precision_score(y_train_fe, y_pred_best_rf_train, average='macro'):.4f}")
print(f"Recall (macro): {recall_score(y_train_fe, y_pred_best_rf_train, average='macro'):.4f}")
print(f"F1-Score (macro): {f1_score(y_train_fe, y_pred_best_rf_train, average='macro'):.4f}")

print("\nClassification Report (Tuned Random Forest, Training Data):")
print(classification_report(y_train_fe, y_pred_best_rf_train))

roc_auc_best_rf_ovr = roc_auc_score(y_train_fe, y_pred_proba_best_rf_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (Tuned Random Forest, macro, One-vs-Rest): {roc_auc_best_rf_ovr:.4f}")

## Generate Predictions on Test Data

Now, with the best performing and tuned Random Forest Classifier, I will generate predictions for the `Credit_Score` on the unseen `df_test_fe` dataset.

In [17]:
# Predict Credit Scores on the test data using the best Random Forest model
test_predictions = best_rf_model.predict(X_test_fe)

# Convert numerical predictions back to original labels if desired for interpretation
# (Poor: 0, Standard: 1, Good: 2)
inverse_credit_score_mapping = {0: 'Poor', 1: 'Standard', 2: 'Good'}
predicted_credit_scores_labels = pd.Series(test_predictions).map(inverse_credit_score_mapping)

print("Predictions on the test set generated.")
print("\nFirst 10 predictions for the test set:")
print(predicted_credit_scores_labels.head(10))

# Add predictions to the original df_test for submission or further analysis
df_test_copy = df_test.copy()
df_test_copy['Credit_Score_Predicted'] = predicted_credit_scores_labels
display(df_test_copy[['ID', 'Credit_Score_Predicted']].head())

NameError: name 'best_rf_model' is not defined

## Hyperparameter Tuning for Random Forest Classifier

Given that the Random Forest Classifier showed the best cross-validated performance, let's now optimize its hyperparameters using `GridSearchCV`. This will help us find the best combination of parameters for improved generalization and performance.

In [18]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define the parameter grid to search
param_grid = {
    'n_estimators': [100, 200, 300], # Number of trees in the forest
    'max_features': ['sqrt', 'log2'], # Number of features to consider at each split
    'max_depth': [10, 20, 30], # Maximum depth of the tree
    'min_samples_split': [2, 5], # Minimum number of samples required to split a node
    'min_samples_leaf': [1, 2], # Minimum number of samples required at a leaf node
    'class_weight': ['balanced'] # Keep 'balanced' as determined before
}

# Initialize the Random Forest Classifier
rf_base = RandomForestClassifier(random_state=42)

# Initialize GridSearchCV
# We'll use the StratifiedKFold (skf) from the previous cross-validation step
# scoring='roc_auc_ovr' to optimize for ROC-AUC
# n_jobs=-1 to use all available cores
# verbose=2 for detailed output
grid_search = GridSearchCV(estimator=rf_base, param_grid=param_grid,
                           cv=skf, n_jobs=-1, verbose=2, scoring='roc_auc_ovr')

print("Starting GridSearchCV for Random Forest. This may take a while...")

# Fit GridSearchCV to the data
# Assuming X_train_fe and y_train_fe are available from previous steps
grid_search.fit(X_train_fe, y_train_fe)

print("GridSearchCV complete.")

# Get the best parameters and best score
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"\nBest parameters found: {best_params}")
print(f"Best ROC-AUC score found during tuning: {best_score:.4f}")

# Train the Random Forest model with the best parameters
best_rf_model = RandomForestClassifier(**best_params, random_state=42)
best_rf_model.fit(X_train_fe, y_train_fe)

print("Random Forest model trained with best parameters.")

NameError: name 'skf' is not defined

## Evaluating the Tuned Random Forest Model

Now that we have tuned the Random Forest Classifier and trained it with the best parameters, let's evaluate its performance on the training data one last time to see the impact of tuning. This will be followed by making predictions on the test set.

In [ ]:
from sklearn.metrics import classification_report

# Predict on the training data using the best Random Forest model
y_pred_best_rf_train = best_rf_model.predict(X_train_fe)
y_pred_proba_best_rf_train = best_rf_model.predict_proba(X_train_fe)

# Evaluate the best Random Forest model on the training data
print("\n--- Tuned Random Forest Model Evaluation on Training Data ---")
print(f"Accuracy: {accuracy_score(y_train_fe, y_pred_best_rf_train):.4f}")
print(f"Precision (macro): {precision_score(y_train_fe, y_pred_best_rf_train, average='macro'):.4f}")
print(f"Recall (macro): {recall_score(y_train_fe, y_pred_best_rf_train, average='macro'):.4f}")
print(f"F1-Score (macro): {f1_score(y_train_fe, y_pred_best_rf_train, average='macro'):.4f}")

print("\nClassification Report (Tuned Random Forest, Training Data):")
print(classification_report(y_train_fe, y_pred_best_rf_train))

roc_auc_best_rf_ovr = roc_auc_score(y_train_fe, y_pred_proba_best_rf_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (Tuned Random Forest, macro, One-vs-Rest): {roc_auc_best_rf_ovr:.4f}")

## Generate Predictions on Test Data

Now, with the best performing and tuned Random Forest Classifier, I will generate predictions for the `Credit_Score` on the unseen `df_test_fe` dataset.

In [ ]:
# Predict Credit Scores on the test data using the best Random Forest model
test_predictions = best_rf_model.predict(X_test_fe)

# Convert numerical predictions back to original labels if desired for interpretation
# (Poor: 0, Standard: 1, Good: 2)
inverse_credit_score_mapping = {0: 'Poor', 1: 'Standard', 2: 'Good'}
predicted_credit_scores_labels = pd.Series(test_predictions).map(inverse_credit_score_mapping)

print("Predictions on the test set generated.")
print("\nFirst 10 predictions for the test set:")
print(predicted_credit_scores_labels.head(10))

# Add predictions to the original df_test for submission or further analysis
df_test_copy = df_test.copy()
df_test_copy['Credit_Score_Predicted'] = predicted_credit_scores_labels
display(df_test_copy[['ID', 'Credit_Score_Predicted']].head())

## Hyperparameter Tuning for Random Forest Classifier

Given that the Random Forest Classifier showed the best cross-validated performance, let's now optimize its hyperparameters using `GridSearchCV`. This will help us find the best combination of parameters for improved generalization and performance.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define the parameter grid to search
param_grid = {
    'n_estimators': [100, 200, 300], # Number of trees in the forest
    'max_features': ['sqrt', 'log2'], # Number of features to consider at each split
    'max_depth': [10, 20, 30], # Maximum depth of the tree
    'min_samples_split': [2, 5], # Minimum number of samples required to split a node
    'min_samples_leaf': [1, 2], # Minimum number of samples required at a leaf node
    'class_weight': ['balanced'] # Keep 'balanced' as determined before
}

# Initialize the Random Forest Classifier
rf_base = RandomForestClassifier(random_state=42)

# Initialize GridSearchCV
# We'll use the StratifiedKFold (skf) from the previous cross-validation step
# scoring='roc_auc_ovr' to optimize for ROC-AUC
# n_jobs=-1 to use all available cores
# verbose=2 for detailed output
grid_search = GridSearchCV(estimator=rf_base, param_grid=param_grid,
                           cv=skf, n_jobs=-1, verbose=2, scoring='roc_auc_ovr')

print("Starting GridSearchCV for Random Forest. This may take a while...")

# Fit GridSearchCV to the data
# Assuming X_train_fe and y_train_fe are available from previous steps
grid_search.fit(X_train_fe, y_train_fe)

print("GridSearchCV complete.")

# Get the best parameters and best score
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"\nBest parameters found: {best_params}")
print(f"Best ROC-AUC score found during tuning: {best_score:.4f}")

# Train the Random Forest model with the best parameters
best_rf_model = RandomForestClassifier(**best_params, random_state=42)
best_rf_model.fit(X_train_fe, y_train_fe)

print("Random Forest model trained with best parameters.")

## Evaluating the Tuned Random Forest Model

Now that we have tuned the Random Forest Classifier and trained it with the best parameters, let's evaluate its performance on the training data one last time to see the impact of tuning. This will be followed by making predictions on the test set.

In [19]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

# Predict on the training data using the best Random Forest model
y_pred_best_rf_train = best_rf_model.predict(X_train_fe)
y_pred_proba_best_rf_train = best_rf_model.predict_proba(X_train_fe)

# Evaluate the best Random Forest model on the training data
print("\n--- Tuned Random Forest Model Evaluation on Training Data ---")
print(f"Accuracy: {accuracy_score(y_train_fe, y_pred_best_rf_train):.4f}")
print(f"Precision (macro): {precision_score(y_train_fe, y_pred_best_rf_train, average='macro'):.4f}")
print(f"Recall (macro): {recall_score(y_train_fe, y_pred_best_rf_train, average='macro'):.4f}")
print(f"F1-Score (macro): {f1_score(y_train_fe, y_pred_best_rf_train, average='macro'):.4f}")

print("\nClassification Report (Tuned Random Forest, Training Data):")
print(classification_report(y_train_fe, y_pred_best_rf_train))

roc_auc_best_rf_ovr = roc_auc_score(y_train_fe, y_pred_proba_best_rf_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (Tuned Random Forest, macro, One-vs-Rest): {roc_auc_best_rf_ovr:.4f}")

NameError: name 'best_rf_model' is not defined

## Generate Predictions on Test Data

Now, with the best performing and tuned Random Forest Classifier, I will generate predictions for the `Credit_Score` on the unseen `df_test_fe` dataset.

In [ ]:
import pandas as pd

# Predict Credit Scores on the test data using the best Random Forest model
test_predictions = best_rf_model.predict(X_test_fe)

# Convert numerical predictions back to original labels if desired for interpretation
# (Poor: 0, Standard: 1, Good: 2)
inverse_credit_score_mapping = {0: 'Poor', 1: 'Standard', 2: 'Good'}
predicted_credit_scores_labels = pd.Series(test_predictions).map(inverse_credit_score_mapping)

print("Predictions on the test set generated.")
print("\nFirst 10 predictions for the test set:")
print(predicted_credit_scores_labels.head(10))

# Add predictions to the original df_test for submission or further analysis
df_test_copy = df_test.copy()
df_test_copy['Credit_Score_Predicted'] = predicted_credit_scores_labels
display(df_test_copy[['ID', 'Credit_Score_Predicted']].head())

## Implementing Cross-Validation for Robust Model Evaluation

To get a more reliable estimate of our models' performance and to mitigate the risk of overfitting, I will now implement 5-fold cross-validation for each model. This will involve splitting the training data into 5 folds, training the model on 4 folds, and validating on the remaining fold, repeating this 5 times, and averaging the results.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import numpy as np
import pandas as pd

# Define the models
models = {
    "Logistic Regression": LogisticRegression(solver='liblinear', random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10) # Using max_depth from previous run
}

# Define scoring metrics
scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc_ovr']

# Initialize Stratified K-Fold Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}

print("--- Starting Cross-Validation ---")

# Assuming X_train_fe and y_train_fe are available from the previous re-execution of the data prep cell.

for name, model in models.items():
    print(f"\nEvaluating {name}...")
    cv_results = cross_validate(model, X_train_fe, y_train_fe, cv=skf, scoring=scoring, n_jobs=-1, verbose=0)

    results[name] = {
        'Accuracy': np.mean(cv_results['test_accuracy']),
        'Precision': np.mean(cv_results['test_precision_macro']),
        'Recall': np.mean(cv_results['test_recall_macro']),
        'F1-Score': np.mean(cv_results['test_f1_macro']),
        'ROC-AUC': np.mean(cv_results['test_roc_auc_ovr'])
    }

    print(f"{name} - Mean Accuracy: {results[name]['Accuracy']:.4f} (+/- {np.std(cv_results['test_accuracy']):.4f})")
    print(f"{name} - Mean Precision: {results[name]['Precision']:.4f} (+/- {np.std(cv_results['test_precision_macro']):.4f})")
    print(f"{name} - Mean Recall: {results[name]['Recall']:.4f} (+/- {np.std(cv_results['test_recall_macro']):.4f})")
    print(f"{name} - Mean F1-Score: {results[name]['F1-Score']:.4f} (+/- {np.std(cv_results['test_f1_macro']):.4f})")
    print(f"{name} - Mean ROC-AUC: {results[name]['ROC-AUC']:.4f} (+/- {np.std(cv_results['test_roc_auc_ovr']):.4f})")

print("\n--- Cross-Validation Complete ---")

# Optionally, display all results in a DataFrame for easier comparison
results_df = pd.DataFrame(results).T
print("\nSummary of Cross-Validation Results:")
display(results_df)

## Re-establishing Data Pipeline and Model Training

It seems that the kernel state might have been reset, leading to undefined variables. I will re-run the entire data preparation pipeline and re-initialize the models to ensure all necessary DataFrames (`df_train`, `df_test`, `df_train_fe`, `df_test_fe`) and feature sets (`X_train`, `y_train`, `X_test`, `X_train_fe`, `y_train_fe`, `X_test_fe`) are correctly defined.

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

print("--- Step 1: Loading Datasets ---")
try:
    df_train = pd.read_csv('/content/train.csv.zip', compression='zip')
    print("Train data loaded successfully.")
except FileNotFoundError:
    print("Error: train.csv.zip not found. Please ensure the file is in /content/.")
except Exception as e:
    print(f"An error occurred while loading train data: {e}")

try:
    df_test = pd.read_csv('/content/test.csv.zip', compression='zip')
    print("Test data loaded successfully.")
except FileNotFoundError:
    print("Error: test.csv.zip not found. Please ensure the file is in /content/.")
except Exception as e:
    print(f"An error occurred while loading test data: {e}")


print("\n--- Step 2: Data Cleaning and Type Conversion ---")
def clean_and_convert_data(df):
    # Convert 'Credit_Score' to numerical using Label Encoding (Poor: 0, Standard: 1, Good: 2)
    credit_score_mapping = {'Poor': 0, 'Standard': 1, 'Good': 2}
    if 'Credit_Score' in df.columns:
        df['Credit_Score'] = df['Credit_Score'].map(credit_score_mapping)

    # Clean and convert 'Age' column
    def clean_age(age_val):
        if isinstance(age_val, str):
            age_val = age_val.replace('_', '').replace('+', '')
            if age_val.isdigit():
                age = int(age_val)
                if 0 < age < 100:
                    return age
                elif age < 0:
                    return abs(age)
                else:
                    return np.nan
            else:
                return np.nan
        return age_val

    df['Age'] = df['Age'].apply(clean_age)
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    df.loc[df['Age'] > 100, 'Age'] = np.nan

    # Clean and convert financial object columns to numeric
    financial_cols = [
        'Annual_Income', 'Num_of_Loan', 'Outstanding_Debt',
        'Amount_invested_monthly', 'Monthly_Balance'
    ]
    for col in financial_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(r'[^\\d.]', '', regex=True)
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Clean and convert 'Credit_History_Age' to years
    def convert_credit_history_age(age_str):
        if isinstance(age_str, str):
            parts = age_str.replace('Years', '').replace('Months', '').replace('and', '').split()
            years = 0
            months = 0
            if len(parts) >= 2:
                try:
                    years = int(parts[0])
                except ValueError:
                    pass
                try:
                    months = int(parts[1])
                except ValueError:
                    pass
            return years + months / 12
        return np.nan

    if 'Credit_History_Age' in df.columns:
        df['Credit_History_Age'] = df['Credit_History_Age'].apply(convert_credit_history_age)

    # Handle problematic '-1' in 'Num_Bank_Accounts' and 'Num_of_Loan' by replacing with NaN
    for col in ['Num_Bank_Accounts', 'Num_of_Loan']:
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan

    return df

df_train = clean_and_convert_data(df_train)
df_test = clean_and_convert_data(df_test)
print("Data cleaning and conversion complete.")


print("\n--- Step 3: Handling Missing Values ---")
numeric_cols_to_impute = [
    'Age', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_of_Loan',
    'Num_Credit_Inquiries', 'Credit_History_Age', 'Amount_invested_monthly', 'Monthly_Balance'
]
categorical_cols_to_impute = [
    'Name', 'Type_of_Loan', 'Num_of_Delayed_Payment'
]

for col in numeric_cols_to_impute:
    if col in df_train.columns:
        median_val_train = df_train[col].median()
        df_train[col].fillna(median_val_train, inplace=True)
    if col in df_test.columns:
        median_val_test = df_test[col].median()
        df_test[col].fillna(median_val_test, inplace=True)

for col in categorical_cols_to_impute:
    if col in df_train.columns:
        mode_val_train = df_train[col].mode()[0]
        df_train[col].fillna(mode_val_train, inplace=True)
    if col in df_test.columns:
        mode_val_test = df_test[col].mode()[0]
        df_test[col].fillna(mode_val_test, inplace=True)
print("Missing value imputation complete.")


print("\n--- Step 4: Categorical Feature Encoding ---")
categorical_cols = [col for col in df_train.select_dtypes(include='object').columns
                    if col not in ['ID', 'Customer_ID', 'SSN']]
df_train = pd.get_dummies(df_train, columns=categorical_cols, drop_first=True)
df_test = pd.get_dummies(df_test, columns=categorical_cols, drop_first=True)

train_cols = set(df_train.columns)
test_cols = set(df_test.columns)

missing_in_test = list(train_cols - test_cols)
for col in missing_in_test:
    if col != 'Credit_Score':
        df_test[col] = 0

missing_in_train = list(test_cols - train_cols)
for col in missing_in_train:
    df_train[col] = 0

df_test = df_test[df_train.drop(columns=['Credit_Score'], errors='ignore').columns]
print("Categorical encoding complete.")


print("\n--- Step 5: Feature Engineering ---")
def create_financial_features(df):
    df_copy = df.copy()
    numeric_cols_for_fe = [
        'Annual_Income', 'Monthly_Inhand_Salary', 'Outstanding_Debt',
        'Total_EMI_per_month', 'Monthly_Balance', 'Credit_Utilization_Ratio'
    ]
    for col in numeric_cols_for_fe:
        if col in df_copy.columns:
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')
            if df_copy[col].isnull().any():
                df_copy[col].fillna(df_copy[col].median(), inplace=True)

    df_copy['Annual_Income_adj'] = df_copy['Annual_Income'].replace(0, df_copy['Annual_Income'].median())
    df_copy['Debt_to_Income_Ratio'] = df_copy['Outstanding_Debt'] / df_copy['Annual_Income_adj']
    df_copy.drop('Annual_Income_adj', axis=1, inplace=True)

    df_copy['Monthly_Inhand_Salary_adj'] = df_copy['Monthly_Inhand_Salary'].replace(0, df_copy['Monthly_Inhand_Salary'].median())
    df_copy['Payment_to_Income_Ratio'] = df_copy['Total_EMI_per_month'] / df_copy['Monthly_Inhand_Salary_adj']
    df_copy.drop('Monthly_Inhand_Salary_adj', axis=1, inplace=True)

    df_copy['Annual_Income_adj'] = df_copy['Annual_Income'].replace(0, df_copy['Annual_Income'].median())
    df_copy['Monthly_Income_Annual_Ratio'] = (df_copy['Monthly_Inhand_Salary'] * 12) / df_copy['Annual_Income_adj']
    df_copy.drop('Annual_Income_adj', axis=1, inplace=True)

    df_copy['Monthly_Inhand_Salary_adj'] = df_copy['Monthly_Inhand_Salary'].replace(0, df_copy['Monthly_Inhand_Salary'].median())
    df_copy['Balance_to_Income_Ratio'] = df_copy['Monthly_Balance'] / df_copy['Monthly_Inhand_Salary_adj']
    df_copy.drop('Monthly_Inhand_Salary_adj', axis=1, inplace=True)

    for col in ['Debt_to_Income_Ratio', 'Payment_to_Income_Ratio', 'Monthly_Income_Annual_Ratio', 'Balance_to_Income_Ratio']:
        if col in df_copy.columns:
            df_copy[col].fillna(df_copy[col].median(), inplace=True)

    return df_copy

df_train_fe = create_financial_features(df_train)
df_test_fe = create_financial_features(df_test)
print("Feature engineering complete. DataFrames: df_train_fe, df_test_fe created.")


print("\n--- Step 6: Prepare Data for Modeling (Pre-Feature Engineering) ---")
non_feature_cols = ['ID', 'Customer_ID', 'SSN']
X_train = df_train.drop(columns=['Credit_Score'] + [col for col in non_feature_cols if col in df_train.columns], errors='ignore')
y_train = df_train['Credit_Score']
X_test = df_test.drop(columns=[col for col in non_feature_cols if col in df_test.columns], errors='ignore')

# Align columns for X_test (pre-FE)
missing_cols_pre_fe = set(X_train.columns) - set(X_test.columns)
for c in missing_cols_pre_fe:
    X_test[c] = 0
X_test = X_test[X_train.columns]

print("X_train, y_train, X_test (pre-FE) prepared.")


print("\n--- Step 7: Train and Evaluate Logistic Regression (Original) ---")
model = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
model.fit(X_train, y_train)
y_pred_train = model.predict(X_train)
y_pred_proba_train = model.predict_proba(X_train)
print(f"Accuracy (LR): {accuracy_score(y_train, y_pred_train):.4f}")
roc_auc_ovr = roc_auc_score(y_train, y_pred_proba_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (LR): {roc_auc_ovr:.4f}")


print("\n--- Step 8: Train and Evaluate Random Forest ---")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)
y_pred_rf_train = rf_model.predict(X_train)
y_pred_proba_rf_train = rf_model.predict_proba(X_train)
print(f"Accuracy (RF): {accuracy_score(y_train, y_pred_rf_train):.4f}")
roc_auc_rf_ovr = roc_auc_score(y_train, y_pred_proba_rf_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (RF): {roc_auc_rf_ovr:.4f}")


print("\n--- Step 9: Train and Evaluate Decision Tree ---")
dt_model = DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10)
dt_model.fit(X_train, y_train)
y_pred_dt_train = dt_model.predict(X_train)
y_pred_proba_dt_train = dt_model.predict_proba(X_train)
print(f"Accuracy (DT): {accuracy_score(y_train, y_pred_dt_train):.4f}")
roc_auc_dt_ovr = roc_auc_score(y_train, y_pred_proba_dt_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (DT): {roc_auc_ovr:.4f}")


print("\n--- Step 10: Prepare Data for Modeling (Post-Feature Engineering) ---")
non_feature_cols = ['ID', 'Customer_ID', 'SSN']
X_train_fe = df_train_fe.drop(columns=['Credit_Score'] + [col for col in non_feature_cols if col in df_train_fe.columns], errors='ignore')
y_train_fe = df_train_fe['Credit_Score']
X_test_fe = df_test_fe.drop(columns=[col for col in non_feature_cols if col in df_test_fe.columns], errors='ignore')

# Align columns for X_test_fe (post-FE)
train_fe_cols = set(X_train_fe.columns)
test_fe_cols = set(X_test_fe.columns)

missing_in_test_fe = list(train_fe_cols - test_fe_cols)
for col in missing_in_test_fe:
    X_test_fe[col] = 0

missing_in_train_fe = list(test_fe_cols - train_fe_cols)
for col in missing_in_train_fe:
    X_train_fe[col] = 0

X_test_fe = X_test_fe[X_train_fe.columns]
print("X_train_fe, y_train_fe, X_test_fe (post-FE) prepared.")


print("\n--- Step 11: Train and Evaluate Logistic Regression (with FE) ---")
lr_model_fe = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
lr_model_fe.fit(X_train_fe, y_train_fe)
y_pred_lr_fe_train = lr_model_fe.predict(X_train_fe)
y_pred_proba_lr_fe_train = lr_model_fe.predict_proba(X_train_fe)
print(f"Accuracy (LR with FE): {accuracy_score(y_train_fe, y_pred_lr_fe_train):.4f}")
roc_auc_lr_fe_ovr = roc_auc_score(y_train_fe, y_pred_proba_lr_fe_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (LR with FE): {roc_auc_lr_fe_ovr:.4f}")

# Initialize Stratified K-Fold Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("\nStratifiedKFold (skf) initialized.")
print("All data preparation and initial model training/evaluation steps re-executed.")

--- Step 1: Loading Datasets ---
Error: train.csv.zip not found. Please ensure the file is in /content/.
Error: test.csv.zip not found. Please ensure the file is in /content/.

--- Step 2: Data Cleaning and Type Conversion ---


NameError: name 'df_train' is not defined

**Reasoning**:
Re-executing the cell `5776e7e5` to ensure all data loading, cleaning, feature engineering, and initial model training/evaluation steps are completed and all necessary variables are defined for subsequent steps.



In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

print("--- Step 1: Loading Datasets ---")
try:
    df_train = pd.read_csv('/content/train.csv.zip', compression='zip')
    print("Train data loaded successfully.")
except FileNotFoundError:
    print("Error: train.csv.zip not found. Please ensure the file is in /content/.")
except Exception as e:
    print(f"An error occurred while loading train data: {e}")

try:
    df_test = pd.read_csv('/content/test.csv.zip', compression='zip')
    print("Test data loaded successfully.")
except FileNotFoundError:
    print("Error: test.csv.zip not found. Please ensure the file is in /content/.")
except Exception as e:
    print(f"An error occurred while loading test data: {e}")


print("\n--- Step 2: Data Cleaning and Type Conversion ---")
def clean_and_convert_data(df):
    # Convert 'Credit_Score' to numerical using Label Encoding (Poor: 0, Standard: 1, Good: 2)
    credit_score_mapping = {'Poor': 0, 'Standard': 1, 'Good': 2}
    if 'Credit_Score' in df.columns:
        df['Credit_Score'] = df['Credit_Score'].map(credit_score_mapping)

    # Clean and convert 'Age' column
    def clean_age(age_val):
        if isinstance(age_val, str):
            age_val = age_val.replace('_', '').replace('+', '')
            if age_val.isdigit():
                age = int(age_val)
                if 0 < age < 100:
                    return age
                elif age < 0:
                    return abs(age)
                else:
                    return np.nan
            else:
                return np.nan
        return age_val

    df['Age'] = df['Age'].apply(clean_age)
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    df.loc[df['Age'] > 100, 'Age'] = np.nan

    # Clean and convert financial object columns to numeric
    financial_cols = [
        'Annual_Income', 'Num_of_Loan', 'Outstanding_Debt',
        'Amount_invested_monthly', 'Monthly_Balance'
    ]
    for col in financial_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(r'[^\\d.]', '', regex=True)
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Clean and convert 'Credit_History_Age' to years
    def convert_credit_history_age(age_str):
        if isinstance(age_str, str):
            parts = age_str.replace('Years', '').replace('Months', '').replace('and', '').split()
            years = 0
            months = 0
            if len(parts) >= 2:
                try:
                    years = int(parts[0])
                except ValueError:
                    pass
                try:
                    months = int(parts[1])
                except ValueError:
                    pass
            return years + months / 12
        return np.nan

    if 'Credit_History_Age' in df.columns:
        df['Credit_History_Age'] = df['Credit_History_Age'].apply(convert_credit_history_age)

    # Handle problematic '-1' in 'Num_Bank_Accounts' and 'Num_of_Loan' by replacing with NaN
    for col in ['Num_Bank_Accounts', 'Num_of_Loan']:
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan

    return df

df_train = clean_and_convert_data(df_train)
df_test = clean_and_convert_data(df_test)
print("Data cleaning and conversion complete.")


print("\n--- Step 3: Handling Missing Values ---")
numeric_cols_to_impute = [
    'Age', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_of_Loan',
    'Num_Credit_Inquiries', 'Credit_History_Age', 'Amount_invested_monthly', 'Monthly_Balance'
]
categorical_cols_to_impute = [
    'Name', 'Type_of_Loan', 'Num_of_Delayed_Payment'
]

for col in numeric_cols_to_impute:
    if col in df_train.columns:
        median_val_train = df_train[col].median()
        df_train[col].fillna(median_val_train, inplace=True)
    if col in df_test.columns:
        median_val_test = df_test[col].median()
        df_test[col].fillna(median_val_test, inplace=True)

for col in categorical_cols_to_impute:
    if col in df_train.columns:
        mode_val_train = df_train[col].mode()[0]
        df_train[col].fillna(mode_val_train, inplace=True)
    if col in df_test.columns:
        mode_val_test = df_test[col].mode()[0]
        df_test[col].fillna(mode_val_test, inplace=True)
print("Missing value imputation complete.")


print("\n--- Step 4: Categorical Feature Encoding ---")
categorical_cols = [col for col in df_train.select_dtypes(include='object').columns
                    if col not in ['ID', 'Customer_ID', 'SSN']]
df_train = pd.get_dummies(df_train, columns=categorical_cols, drop_first=True)
df_test = pd.get_dummies(df_test, columns=categorical_cols, drop_first=True)

train_cols = set(df_train.columns)
test_cols = set(df_test.columns)

missing_in_test = list(train_cols - test_cols)
for col in missing_in_test:
    if col != 'Credit_Score':
        df_test[col] = 0

missing_in_train = list(test_cols - train_cols)
for col in missing_in_train:
    df_train[col] = 0

df_test = df_test[df_train.drop(columns=['Credit_Score'], errors='ignore').columns]
print("Categorical encoding complete.")


print("\n--- Step 5: Feature Engineering ---")
def create_financial_features(df):
    df_copy = df.copy()
    numeric_cols_for_fe = [
        'Annual_Income', 'Monthly_Inhand_Salary', 'Outstanding_Debt',
        'Total_EMI_per_month', 'Monthly_Balance', 'Credit_Utilization_Ratio'
    ]
    for col in numeric_cols_for_fe:
        if col in df_copy.columns:
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')
            if df_copy[col].isnull().any():
                df_copy[col].fillna(df_copy[col].median(), inplace=True)

    df_copy['Annual_Income_adj'] = df_copy['Annual_Income'].replace(0, df_copy['Annual_Income'].median())
    df_copy['Debt_to_Income_Ratio'] = df_copy['Outstanding_Debt'] / df_copy['Annual_Income_adj']
    df_copy.drop('Annual_Income_adj', axis=1, inplace=True)

    df_copy['Monthly_Inhand_Salary_adj'] = df_copy['Monthly_Inhand_Salary'].replace(0, df_copy['Monthly_Inhand_Salary'].median())
    df_copy['Payment_to_Income_Ratio'] = df_copy['Total_EMI_per_month'] / df_copy['Monthly_Inhand_Salary_adj']
    df_copy.drop('Monthly_Inhand_Salary_adj', axis=1, inplace=True)

    df_copy['Annual_Income_adj'] = df_copy['Annual_Income'].replace(0, df_copy['Annual_Income'].median())
    df_copy['Monthly_Income_Annual_Ratio'] = (df_copy['Monthly_Inhand_Salary'] * 12) / df_copy['Annual_Income_adj']
    df_copy.drop('Annual_Income_adj', axis=1, inplace=True)

    df_copy['Monthly_Inhand_Salary_adj'] = df_copy['Monthly_Inhand_Salary'].replace(0, df_copy['Monthly_Inhand_Salary'].median())
    df_copy['Balance_to_Income_Ratio'] = df_copy['Monthly_Balance'] / df_copy['Monthly_Inhand_Salary_adj']
    df_copy.drop('Monthly_Inhand_Salary_adj', axis=1, inplace=True)

    for col in ['Debt_to_Income_Ratio', 'Payment_to_Income_Ratio', 'Monthly_Income_Annual_Ratio', 'Balance_to_Income_Ratio']:
        if col in df_copy.columns:
            df_copy[col].fillna(df_copy[col].median(), inplace=True)

    return df_copy

df_train_fe = create_financial_features(df_train)
df_test_fe = create_financial_features(df_test)
print("Feature engineering complete. DataFrames: df_train_fe, df_test_fe created.")


print("\n--- Step 6: Prepare Data for Modeling (Pre-Feature Engineering) ---")
non_feature_cols = ['ID', 'Customer_ID', 'SSN']
X_train = df_train.drop(columns=['Credit_Score'] + [col for col in non_feature_cols if col in df_train.columns], errors='ignore')
y_train = df_train['Credit_Score']
X_test = df_test.drop(columns=[col for col in non_feature_cols if col in df_test.columns], errors='ignore')

# Align columns for X_test (pre-FE)
missing_cols_pre_fe = set(X_train.columns) - set(X_test.columns)
for c in missing_cols_pre_fe:
    X_test[c] = 0
X_test = X_test[X_train.columns]

print("X_train, y_train, X_test (pre-FE) prepared.")


print("\n--- Step 7: Train and Evaluate Logistic Regression (Original) ---")
model = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
model.fit(X_train, y_train)
y_pred_train = model.predict(X_train)
y_pred_proba_train = model.predict_proba(X_train)
print(f"Accuracy (LR): {accuracy_score(y_train, y_pred_train):.4f}")
roc_auc_ovr = roc_auc_score(y_train, y_pred_proba_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (LR): {roc_auc_ovr:.4f}")


print("\n--- Step 8: Train and Evaluate Random Forest ---")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)
y_pred_rf_train = rf_model.predict(X_train)
y_pred_proba_rf_train = rf_model.predict_proba(X_train)
print(f"Accuracy (RF): {accuracy_score(y_train, y_pred_rf_train):.4f}")
roc_auc_rf_ovr = roc_auc_score(y_train, y_pred_proba_rf_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (RF): {roc_auc_rf_ovr:.4f}")


print("\n--- Step 9: Train and Evaluate Decision Tree ---")
dt_model = DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10)
dt_model.fit(X_train, y_train)
y_pred_dt_train = dt_model.predict(X_train)
y_pred_proba_dt_train = dt_model.predict_proba(X_train)
print(f"Accuracy (DT): {accuracy_score(y_train, y_pred_dt_train):.4f}")
roc_auc_dt_ovr = roc_auc_score(y_train, y_pred_proba_dt_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (DT): {roc_auc_ovr:.4f}")


print("\n--- Step 10: Prepare Data for Modeling (Post-Feature Engineering) ---")
non_feature_cols = ['ID', 'Customer_ID', 'SSN']
X_train_fe = df_train_fe.drop(columns=['Credit_Score'] + [col for col in non_feature_cols if col in df_train_fe.columns], errors='ignore')
y_train_fe = df_train_fe['Credit_Score']
X_test_fe = df_test_fe.drop(columns=[col for col in non_feature_cols if col in df_test_fe.columns], errors='ignore')

# Align columns for X_test_fe (post-FE)
train_fe_cols = set(X_train_fe.columns)
test_fe_cols = set(X_test_fe.columns)

missing_in_test_fe = list(train_fe_cols - test_fe_cols)
for col in missing_in_test_fe:
    X_test_fe[col] = 0

missing_in_train_fe = list(test_fe_cols - train_fe_cols)
for col in missing_in_train_fe:
    X_train_fe[col] = 0

X_test_fe = X_test_fe[X_train_fe.columns]
print("X_train_fe, y_train_fe, X_test_fe (post-FE) prepared.")


print("\n--- Step 11: Train and Evaluate Logistic Regression (with FE) ---")
lr_model_fe = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
lr_model_fe.fit(X_train_fe, y_train_fe)
y_pred_lr_fe_train = lr_model_fe.predict(X_train_fe)
y_pred_proba_lr_fe_train = lr_model_fe.predict_proba(X_train_fe)
print(f"Accuracy (LR with FE): {accuracy_score(y_train_fe, y_pred_lr_fe_train):.4f}")
roc_auc_lr_fe_ovr = roc_auc_score(y_train_fe, y_pred_proba_lr_fe_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (LR with FE): {roc_auc_lr_fe_ovr:.4f}")

# Initialize Stratified K-Fold Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("\nStratifiedKFold (skf) initialized.")
print("All data preparation and initial model training/evaluation steps re-executed.")

# Task
The user wants to predict credit scores based on a dataset containing various financial and personal information. The task involves loading and preprocessing the data, handling missing values, encoding categorical features, engineering new features, training multiple classification models (Logistic Regression, Random Forest, Decision Tree), evaluating their performance, and then tuning the best performing model using GridSearchCV. Finally, the best tuned model will be used to generate predictions on the test dataset.

## Evaluate models with cross-validation

### Subtask:
Implement 5-fold stratified cross-validation to evaluate Logistic Regression, Random Forest, and Decision Tree models on the training data with feature engineered features (X_train_fe, y_train_fe) using multiple scoring metrics.


**Reasoning**:
The previous code cell `5776e7e5` executed successfully, ensuring all necessary dataframes and variables like `X_train_fe`, `y_train_fe`, and `skf` are defined. Now I need to proceed with the cross-validation as per the subtask. The code in cell `1c763fa8` is appropriate for this step.



In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import numpy as np
import pandas as pd

# Define the models
models = {
    "Logistic Regression": LogisticRegression(solver='liblinear', random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10) # Using max_depth from previous run
}

# Define scoring metrics
scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc_ovr']

# Initialize Stratified K-Fold Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}

print("--- Starting Cross-Validation ---")

# Assuming X_train_fe and y_train_fe are available from the previous re-execution of the data prep cell.

for name, model in models.items():
    print(f"\nEvaluating {name}...")
    cv_results = cross_validate(model, X_train_fe, y_train_fe, cv=skf, scoring=scoring, n_jobs=-1, verbose=0)

    results[name] = {
        'Accuracy': np.mean(cv_results['test_accuracy']),
        'Precision': np.mean(cv_results['test_precision_macro']),
        'Recall': np.mean(cv_results['test_recall_macro']),
        'F1-Score': np.mean(cv_results['test_f1_macro']),
        'ROC-AUC': np.mean(cv_results['test_roc_auc_ovr'])
    }

    print(f"{name} - Mean Accuracy: {results[name]['Accuracy']:.4f} (+/- {np.std(cv_results['test_accuracy']):.4f})")
    print(f"{name} - Mean Precision: {results[name]['Precision']:.4f} (+/- {np.std(cv_results['test_precision_macro']):.4f})")
    print(f"{name} - Mean Recall: {results[name]['Recall']:.4f} (+/- {np.std(cv_results['test_recall_macro']):.4f})")
    print(f"{name} - Mean F1-Score: {results[name]['F1-Score']:.4f} (+/- {np.std(cv_results['test_f1_macro']):.4f})")
    print(f"{name} - Mean ROC-AUC: {results[name]['ROC-AUC']:.4f} (+/- {np.std(cv_results['test_roc_auc_ovr']):.4f})")

print("\n--- Cross-Validation Complete ---")

# Optionally, display all results in a DataFrame for easier comparison
results_df = pd.DataFrame(results).T
print("\nSummary of Cross-Validation Results:")
display(results_df)

**Reasoning**:
Execute the cross-validation to evaluate the models on the feature-engineered data, now that `X_train_fe`, `y_train_fe`, and `skf` are expected to be defined after re-running the data preparation cell.



In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import numpy as np
import pandas as pd

# Define the models
models = {
    "Logistic Regression": LogisticRegression(solver='liblinear', random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10) # Using max_depth from previous run
}

# Define scoring metrics
scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc_ovr']

# Initialize Stratified K-Fold Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}

print("--- Starting Cross-Validation ---")

# Assuming X_train_fe and y_train_fe are available from the previous re-execution of the data prep cell.

for name, model in models.items():
    print(f"\nEvaluating {name}...")
    cv_results = cross_validate(model, X_train_fe, y_train_fe, cv=skf, scoring=scoring, n_jobs=-1, verbose=0)

    results[name] = {
        'Accuracy': np.mean(cv_results['test_accuracy']),
        'Precision': np.mean(cv_results['test_precision_macro']),
        'Recall': np.mean(cv_results['test_recall_macro']),
        'F1-Score': np.mean(cv_results['test_f1_macro']),
        'ROC-AUC': np.mean(cv_results['test_roc_auc_ovr'])
    }

    print(f"{name} - Mean Accuracy: {results[name]['Accuracy']:.4f} (+/- {np.std(cv_results['test_accuracy']):.4f})")
    print(f"{name} - Mean Precision: {results[name]['Precision']:.4f} (+/- {np.std(cv_results['test_precision_macro']):.4f})")
    print(f"{name} - Mean Recall: {results[name]['Recall']:.4f} (+/- {np.std(cv_results['test_recall_macro']):.4f})")
    print(f"{name} - Mean F1-Score: {results[name]['F1-Score']:.4f} (+/- {np.std(cv_results['test_f1_macro']):.4f})")
    print(f"{name} - Mean ROC-AUC: {results[name]['ROC-AUC']:.4f} (+/- {np.std(cv_results['test_roc_auc_ovr']):.4f})")

print("\n--- Cross-Validation Complete ---")

# Optionally, display all results in a DataFrame for easier comparison
results_df = pd.DataFrame(results).T
print("\nSummary of Cross-Validation Results:")
display(results_df)

**Reasoning**:
Re-executing the cell `9cacdfe1` to ensure all data loading, cleaning, feature engineering, and initial model training/evaluation steps are completed and all necessary variables are defined for subsequent steps.



In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

print("--- Step 1: Loading Datasets ---")
try:
    df_train = pd.read_csv('/content/train.csv.zip', compression='zip')
    print("Train data loaded successfully.")
except FileNotFoundError:
    print("Error: train.csv.zip not found. Please ensure the file is in /content/.")
except Exception as e:
    print(f"An error occurred while loading train data: {e}")

try:
    df_test = pd.read_csv('/content/test.csv.zip', compression='zip')
    print("Test data loaded successfully.")
except FileNotFoundError:
    print("Error: test.csv.zip not found. Please ensure the file is in /content/.")
except Exception as e:
    print(f"An error occurred while loading test data: {e}")


print("\n--- Step 2: Data Cleaning and Type Conversion ---")
def clean_and_convert_data(df):
    # Convert 'Credit_Score' to numerical using Label Encoding (Poor: 0, Standard: 1, Good: 2)
    credit_score_mapping = {'Poor': 0, 'Standard': 1, 'Good': 2}
    if 'Credit_Score' in df.columns:
        df['Credit_Score'] = df['Credit_Score'].map(credit_score_mapping)

    # Clean and convert 'Age' column
    def clean_age(age_val):
        if isinstance(age_val, str):
            age_val = age_val.replace('_', '').replace('+', '')
            if age_val.isdigit():
                age = int(age_val)
                if 0 < age < 100:
                    return age
                elif age < 0:
                    return abs(age)
                else:
                    return np.nan
            else:
                return np.nan
        return age_val

    df['Age'] = df['Age'].apply(clean_age)
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    df.loc[df['Age'] > 100, 'Age'] = np.nan

    # Clean and convert financial object columns to numeric
    financial_cols = [
        'Annual_Income', 'Num_of_Loan', 'Outstanding_Debt',
        'Amount_invested_monthly', 'Monthly_Balance'
    ]
    for col in financial_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(r'[^\\d.]', '', regex=True)
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Clean and convert 'Credit_History_Age' to years
    def convert_credit_history_age(age_str):
        if isinstance(age_str, str):
            parts = age_str.replace('Years', '').replace('Months', '').replace('and', '').split()
            years = 0
            months = 0
            if len(parts) >= 2:
                try:
                    years = int(parts[0])
                except ValueError:
                    pass
                try:
                    months = int(parts[1])
                except ValueError:
                    pass
            return years + months / 12
        return np.nan

    if 'Credit_History_Age' in df.columns:
        df['Credit_History_Age'] = df['Credit_History_Age'].apply(convert_credit_history_age)

    # Handle problematic '-1' in 'Num_Bank_Accounts' and 'Num_of_Loan' by replacing with NaN
    for col in ['Num_Bank_Accounts', 'Num_of_Loan']:
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan

    return df

df_train = clean_and_convert_data(df_train)
df_test = clean_and_convert_data(df_test)
print("Data cleaning and conversion complete.")


print("\n--- Step 3: Handling Missing Values ---")
numeric_cols_to_impute = [
    'Age', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_of_Loan',
    'Num_Credit_Inquiries', 'Credit_History_Age', 'Amount_invested_monthly', 'Monthly_Balance'
]
categorical_cols_to_impute = [
    'Name', 'Type_of_Loan', 'Num_of_Delayed_Payment'
]

for col in numeric_cols_to_impute:
    if col in df_train.columns:
        median_val_train = df_train[col].median()
        df_train[col].fillna(median_val_train, inplace=True)
    if col in df_test.columns:
        median_val_test = df_test[col].median()
        df_test[col].fillna(median_val_test, inplace=True)

for col in categorical_cols_to_impute:
    if col in df_train.columns:
        mode_val_train = df_train[col].mode()[0]
        df_train[col].fillna(mode_val_train, inplace=True)
    if col in df_test.columns:
        mode_val_test = df_test[col].mode()[0]
        df_test[col].fillna(mode_val_test, inplace=True)
print("Missing value imputation complete.")


print("\n--- Step 4: Categorical Feature Encoding ---")
categorical_cols = [col for col in df_train.select_dtypes(include='object').columns
                    if col not in ['ID', 'Customer_ID', 'SSN']]
df_train = pd.get_dummies(df_train, columns=categorical_cols, drop_first=True)
df_test = pd.get_dummies(df_test, columns=categorical_cols, drop_first=True)

train_cols = set(df_train.columns)
test_cols = set(df_test.columns)

missing_in_test = list(train_cols - test_cols)
for col in missing_in_test:
    if col != 'Credit_Score':
        df_test[col] = 0

missing_in_train = list(test_cols - train_cols)
for col in missing_in_train:
    df_train[col] = 0

df_test = df_test[df_train.drop(columns=['Credit_Score'], errors='ignore').columns]
print("Categorical encoding complete.")


print("\n--- Step 5: Feature Engineering ---")
def create_financial_features(df):
    df_copy = df.copy()
    numeric_cols_for_fe = [
        'Annual_Income', 'Monthly_Inhand_Salary', 'Outstanding_Debt',
        'Total_EMI_per_month', 'Monthly_Balance', 'Credit_Utilization_Ratio'
    ]
    for col in numeric_cols_for_fe:
        if col in df_copy.columns:
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')
            if df_copy[col].isnull().any():
                df_copy[col].fillna(df_copy[col].median(), inplace=True)

    df_copy['Annual_Income_adj'] = df_copy['Annual_Income'].replace(0, df_copy['Annual_Income'].median())
    df_copy['Debt_to_Income_Ratio'] = df_copy['Outstanding_Debt'] / df_copy['Annual_Income_adj']
    df_copy.drop('Annual_Income_adj', axis=1, inplace=True)

    df_copy['Monthly_Inhand_Salary_adj'] = df_copy['Monthly_Inhand_Salary'].replace(0, df_copy['Monthly_Inhand_Salary'].median())
    df_copy['Payment_to_Income_Ratio'] = df_copy['Total_EMI_per_month'] / df_copy['Monthly_Inhand_Salary_adj']
    df_copy.drop('Monthly_Inhand_Salary_adj', axis=1, inplace=True)

    df_copy['Annual_Income_adj'] = df_copy['Annual_Income'].replace(0, df_copy['Annual_Income'].median())
    df_copy['Monthly_Income_Annual_Ratio'] = (df_copy['Monthly_Inhand_Salary'] * 12) / df_copy['Annual_Income_adj']
    df_copy.drop('Annual_Income_adj', axis=1, inplace=True)

    df_copy['Monthly_Inhand_Salary_adj'] = df_copy['Monthly_Inhand_Salary'].replace(0, df_copy['Monthly_Inhand_Salary'].median())
    df_copy['Balance_to_Income_Ratio'] = df_copy['Monthly_Balance'] / df_copy['Monthly_Inhand_Salary_adj']
    df_copy.drop('Monthly_Inhand_Salary_adj', axis=1, inplace=True)

    for col in ['Debt_to_Income_Ratio', 'Payment_to_Income_Ratio', 'Monthly_Income_Annual_Ratio', 'Balance_to_Income_Ratio']:
        if col in df_copy.columns:
            df_copy[col].fillna(df_copy[col].median(), inplace=True)

    return df_copy

df_train_fe = create_financial_features(df_train)
df_test_fe = create_financial_features(df_test)
print("Feature engineering complete. DataFrames: df_train_fe, df_test_fe created.")


print("\n--- Step 6: Prepare Data for Modeling (Pre-Feature Engineering) ---")
non_feature_cols = ['ID', 'Customer_ID', 'SSN']
X_train = df_train.drop(columns=['Credit_Score'] + [col for col in non_feature_cols if col in df_train.columns], errors='ignore')
y_train = df_train['Credit_Score']
X_test = df_test.drop(columns=[col for col in non_feature_cols if col in df_test.columns], errors='ignore')

# Align columns for X_test (pre-FE)
missing_cols_pre_fe = set(X_train.columns) - set(X_test.columns)
for c in missing_cols_pre_fe:
    X_test[c] = 0
X_test = X_test[X_train.columns]

print("X_train, y_train, X_test (pre-FE) prepared.")


print("\n--- Step 7: Train and Evaluate Logistic Regression (Original) ---")
model = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
model.fit(X_train, y_train)
y_pred_train = model.predict(X_train)
y_pred_proba_train = model.predict_proba(X_train)
print(f"Accuracy (LR): {accuracy_score(y_train, y_pred_train):.4f}")
roc_auc_ovr = roc_auc_score(y_train, y_pred_proba_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (LR): {roc_auc_ovr:.4f}")


print("\n--- Step 8: Train and Evaluate Random Forest ---")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)
y_pred_rf_train = rf_model.predict(X_train)
y_pred_proba_rf_train = rf_model.predict_proba(X_train)
print(f"Accuracy (RF): {accuracy_score(y_train, y_pred_rf_train):.4f}")
roc_auc_rf_ovr = roc_auc_score(y_train, y_pred_proba_rf_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (RF): {roc_auc_rf_ovr:.4f}")


print("\n--- Step 9: Train and Evaluate Decision Tree ---")
dt_model = DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10)
dt_model.fit(X_train, y_train)
y_pred_dt_train = dt_model.predict(X_train)
y_pred_proba_dt_train = dt_model.predict_proba(X_train)
print(f"Accuracy (DT): {accuracy_score(y_train, y_pred_dt_train):.4f}")
roc_auc_dt_ovr = roc_auc_score(y_train, y_pred_proba_dt_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (DT): {roc_auc_ovr:.4f}")


print("\n--- Step 10: Prepare Data for Modeling (Post-Feature Engineering) ---")
non_feature_cols = ['ID', 'Customer_ID', 'SSN']
X_train_fe = df_train_fe.drop(columns=['Credit_Score'] + [col for col in non_feature_cols if col in df_train_fe.columns], errors='ignore')
y_train_fe = df_train_fe['Credit_Score']
X_test_fe = df_test_fe.drop(columns=[col for col in non_feature_cols if col in df_test_fe.columns], errors='ignore')

# Align columns for X_test_fe (post-FE)
train_fe_cols = set(X_train_fe.columns)
test_fe_cols = set(X_test_fe.columns)

missing_in_test_fe = list(train_fe_cols - test_fe_cols)
for col in missing_in_test_fe:
    X_test_fe[col] = 0

missing_in_train_fe = list(test_fe_cols - train_fe_cols)
for col in missing_in_train_fe:
    X_train_fe[col] = 0

X_test_fe = X_test_fe[X_train_fe.columns]
print("X_train_fe, y_train_fe, X_test_fe (post-FE) prepared.")


print("\n--- Step 11: Train and Evaluate Logistic Regression (with FE) ---")
lr_model_fe = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
lr_model_fe.fit(X_train_fe, y_train_fe)
y_pred_lr_fe_train = lr_model_fe.predict(X_train_fe)
y_pred_proba_lr_fe_train = lr_model_fe.predict_proba(X_train_fe)
print(f"Accuracy (LR with FE): {accuracy_score(y_train_fe, y_pred_lr_fe_train):.4f}")
roc_auc_lr_fe_ovr = roc_auc_score(y_train_fe, y_pred_proba_lr_fe_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (LR with FE): {roc_auc_lr_fe_ovr:.4f}")

# Initialize Stratified K-Fold Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("\nStratifiedKFold (skf) initialized.")
print("All data preparation and initial model training/evaluation steps re-executed.")

--- Step 1: Loading Datasets ---
Error: train.csv.zip not found. Please ensure the file is in /content/.
Error: test.csv.zip not found. Please ensure the file is in /content/.

--- Step 2: Data Cleaning and Type Conversion ---


NameError: name 'df_train' is not defined

# Task
The user wants to predict credit scores based on a dataset containing various financial and personal information. The task involves loading and preprocessing the data, handling missing values, encoding categorical features, engineering new features, training multiple classification models (Logistic Regression, Random Forest, Decision Tree), evaluating their performance, and then tuning the best performing model using GridSearchCV. Finally, the best tuned model will be used to generate predictions on the test dataset.

## Re-execute Data Preparation and Feature Engineering

### Subtask:
Re-execute the entire data preparation pipeline, including data loading, cleaning, missing value imputation, categorical encoding, and feature engineering, to ensure all necessary dataframes and feature sets (e.g., `df_train`, `df_test`, `df_train_fe`, `df_test_fe`, `X_train_fe`, `y_train_fe`) are correctly defined and available for subsequent steps. Also, ensure the `skf` object is initialized.


**Reasoning**:
Re-executing the specified code cell (`9cacdfe1`) to re-initialize all dataframes and modeling variables as required by the subtask instructions.



In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

print("--- Step 1: Loading Datasets ---")
try:
    df_train = pd.read_csv('/content/train.csv.zip', compression='zip')
    print("Train data loaded successfully.")
except FileNotFoundError:
    print("Error: train.csv.zip not found. Please ensure the file is in /content/.")
except Exception as e:
    print(f"An error occurred while loading train data: {e}")

try:
    df_test = pd.read_csv('/content/test.csv.zip', compression='zip')
    print("Test data loaded successfully.")
except FileNotFoundError:
    print("Error: test.csv.zip not found. Please ensure the file is in /content/.")
except Exception as e:
    print(f"An error occurred while loading test data: {e}")


print("\n--- Step 2: Data Cleaning and Type Conversion ---")
def clean_and_convert_data(df):
    # Convert 'Credit_Score' to numerical using Label Encoding (Poor: 0, Standard: 1, Good: 2)
    credit_score_mapping = {'Poor': 0, 'Standard': 1, 'Good': 2}
    if 'Credit_Score' in df.columns:
        df['Credit_Score'] = df['Credit_Score'].map(credit_score_mapping)

    # Clean and convert 'Age' column
    def clean_age(age_val):
        if isinstance(age_val, str):
            age_val = age_val.replace('_', '').replace('+', '')
            if age_val.isdigit():
                age = int(age_val)
                if 0 < age < 100:
                    return age
                elif age < 0:
                    return abs(age)
                else:
                    return np.nan
            else:
                return np.nan
        return age_val

    df['Age'] = df['Age'].apply(clean_age)
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    df.loc[df['Age'] > 100, 'Age'] = np.nan

    # Clean and convert financial object columns to numeric
    financial_cols = [
        'Annual_Income', 'Num_of_Loan', 'Outstanding_Debt',
        'Amount_invested_monthly', 'Monthly_Balance'
    ]
    for col in financial_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(r'[^\\d.]', '', regex=True)
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Clean and convert 'Credit_History_Age' to years
    def convert_credit_history_age(age_str):
        if isinstance(age_str, str):
            parts = age_str.replace('Years', '').replace('Months', '').replace('and', '').split()
            years = 0
            months = 0
            if len(parts) >= 2:
                try:
                    years = int(parts[0])
                except ValueError:
                    pass
                try:
                    months = int(parts[1])
                except ValueError:
                    pass
            return years + months / 12
        return np.nan

    if 'Credit_History_Age' in df.columns:
        df['Credit_History_Age'] = df['Credit_History_Age'].apply(convert_credit_history_age)

    # Handle problematic '-1' in 'Num_Bank_Accounts' and 'Num_of_Loan' by replacing with NaN
    for col in ['Num_Bank_Accounts', 'Num_of_Loan']:
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan

    return df

df_train = clean_and_convert_data(df_train)
df_test = clean_and_convert_data(df_test)
print("Data cleaning and conversion complete.")


print("\n--- Step 3: Handling Missing Values ---")
numeric_cols_to_impute = [
    'Age', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_of_Loan',
    'Num_Credit_Inquiries', 'Credit_History_Age', 'Amount_invested_monthly', 'Monthly_Balance'
]
categorical_cols_to_impute = [
    'Name', 'Type_of_Loan', 'Num_of_Delayed_Payment'
]

for col in numeric_cols_to_impute:
    if col in df_train.columns:
        median_val_train = df_train[col].median()
        df_train[col].fillna(median_val_train, inplace=True)
    if col in df_test.columns:
        median_val_test = df_test[col].median()
        df_test[col].fillna(median_val_test, inplace=True)

for col in categorical_cols_to_impute:
    if col in df_train.columns:
        mode_val_train = df_train[col].mode()[0]
        df_train[col].fillna(mode_val_train, inplace=True)
    if col in df_test.columns:
        mode_val_test = df_test[col].mode()[0]
        df_test[col].fillna(mode_val_test, inplace=True)
print("Missing value imputation complete.")


print("\n--- Step 4: Categorical Feature Encoding ---")
categorical_cols = [col for col in df_train.select_dtypes(include='object').columns
                    if col not in ['ID', 'Customer_ID', 'SSN']]
df_train = pd.get_dummies(df_train, columns=categorical_cols, drop_first=True)
df_test = pd.get_dummies(df_test, columns=categorical_cols, drop_first=True)

train_cols = set(df_train.columns)
test_cols = set(df_test.columns)

missing_in_test = list(train_cols - test_cols)
for col in missing_in_test:
    if col != 'Credit_Score':
        df_test[col] = 0

missing_in_train = list(test_cols - train_cols)
for col in missing_in_train:
    df_train[col] = 0

df_test = df_test[df_train.drop(columns=['Credit_Score'], errors='ignore').columns]
print("Categorical encoding complete.")


print("\n--- Step 5: Feature Engineering ---")
def create_financial_features(df):
    df_copy = df.copy()
    numeric_cols_for_fe = [
        'Annual_Income', 'Monthly_Inhand_Salary', 'Outstanding_Debt',
        'Total_EMI_per_month', 'Monthly_Balance', 'Credit_Utilization_Ratio'
    ]
    for col in numeric_cols_for_fe:
        if col in df_copy.columns:
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')
            if df_copy[col].isnull().any():
                df_copy[col].fillna(df_copy[col].median(), inplace=True)

    df_copy['Annual_Income_adj'] = df_copy['Annual_Income'].replace(0, df_copy['Annual_Income'].median())
    df_copy['Debt_to_Income_Ratio'] = df_copy['Outstanding_Debt'] / df_copy['Annual_Income_adj']
    df_copy.drop('Annual_Income_adj', axis=1, inplace=True)

    df_copy['Monthly_Inhand_Salary_adj'] = df_copy['Monthly_Inhand_Salary'].replace(0, df_copy['Monthly_Inhand_Salary'].median())
    df_copy['Payment_to_Income_Ratio'] = df_copy['Total_EMI_per_month'] / df_copy['Monthly_Inhand_Salary_adj']
    df_copy.drop('Monthly_Inhand_Salary_adj', axis=1, inplace=True)

    df_copy['Annual_Income_adj'] = df_copy['Annual_Income'].replace(0, df_copy['Annual_Income'].median())
    df_copy['Monthly_Income_Annual_Ratio'] = (df_copy['Monthly_Inhand_Salary'] * 12) / df_copy['Annual_Income_adj']
    df_copy.drop('Annual_Income_adj', axis=1, inplace=True)

    df_copy['Monthly_Inhand_Salary_adj'] = df_copy['Monthly_Inhand_Salary'].replace(0, df_copy['Monthly_Inhand_Salary'].median())
    df_copy['Balance_to_Income_Ratio'] = df_copy['Monthly_Balance'] / df_copy['Monthly_Inhand_Salary_adj']
    df_copy.drop('Monthly_Inhand_Salary_adj', axis=1, inplace=True)

    for col in ['Debt_to_Income_Ratio', 'Payment_to_Income_Ratio', 'Monthly_Income_Annual_Ratio', 'Balance_to_Income_Ratio']:
        if col in df_copy.columns:
            df_copy[col].fillna(df_copy[col].median(), inplace=True)

    return df_copy

df_train_fe = create_financial_features(df_train)
df_test_fe = create_financial_features(df_test)
print("Feature engineering complete. DataFrames: df_train_fe, df_test_fe created.")


print("\n--- Step 6: Prepare Data for Modeling (Pre-Feature Engineering) ---")
non_feature_cols = ['ID', 'Customer_ID', 'SSN']
X_train = df_train.drop(columns=['Credit_Score'] + [col for col in non_feature_cols if col in df_train.columns], errors='ignore')
y_train = df_train['Credit_Score']
X_test = df_test.drop(columns=[col for col in non_feature_cols if col in df_test.columns], errors='ignore')

# Align columns for X_test (pre-FE)
missing_cols_pre_fe = set(X_train.columns) - set(X_test.columns)
for c in missing_cols_pre_fe:
    X_test[c] = 0
X_test = X_test[X_train.columns]

print("X_train, y_train, X_test (pre-FE) prepared.")


print("\n--- Step 7: Train and Evaluate Logistic Regression (Original) ---")
model = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
model.fit(X_train, y_train)
y_pred_train = model.predict(X_train)
y_pred_proba_train = model.predict_proba(X_train)
print(f"Accuracy (LR): {accuracy_score(y_train, y_pred_train):.4f}")
roc_auc_ovr = roc_auc_score(y_train, y_pred_proba_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (LR): {roc_auc_ovr:.4f}")


print("\n--- Step 8: Train and Evaluate Random Forest ---")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)
y_pred_rf_train = rf_model.predict(X_train)
y_pred_proba_rf_train = rf_model.predict_proba(X_train)
print(f"Accuracy (RF): {accuracy_score(y_train, y_pred_rf_train):.4f}")
roc_auc_rf_ovr = roc_auc_score(y_train, y_pred_proba_rf_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (RF): {roc_auc_rf_ovr:.4f}")


print("\n--- Step 9: Train and Evaluate Decision Tree ---")
dt_model = DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10)
dt_model.fit(X_train, y_train)
y_pred_dt_train = dt_model.predict(X_train)
y_pred_proba_dt_train = dt_model.predict_proba(X_train)
print(f"Accuracy (DT): {accuracy_score(y_train, y_pred_dt_train):.4f}")
roc_auc_dt_ovr = roc_auc_score(y_train, y_pred_proba_dt_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (DT): {roc_auc_ovr:.4f}")


print("\n--- Step 10: Prepare Data for Modeling (Post-Feature Engineering) ---")
non_feature_cols = ['ID', 'Customer_ID', 'SSN']
X_train_fe = df_train_fe.drop(columns=['Credit_Score'] + [col for col in non_feature_cols if col in df_train_fe.columns], errors='ignore')
y_train_fe = df_train_fe['Credit_Score']
X_test_fe = df_test_fe.drop(columns=[col for col in non_feature_cols if col in df_test_fe.columns], errors='ignore')

# Align columns for X_test_fe (post-FE)
train_fe_cols = set(X_train_fe.columns)
test_fe_cols = set(X_test_fe.columns)

missing_in_test_fe = list(train_fe_cols - test_fe_cols)
for col in missing_in_test_fe:
    X_test_fe[col] = 0

missing_in_train_fe = list(test_fe_cols - train_fe_cols)
for col in missing_in_train_fe:
    X_train_fe[col] = 0

X_test_fe = X_test_fe[X_train_fe.columns]
print("X_train_fe, y_train_fe, X_test_fe (post-FE) prepared.")


print("\n--- Step 11: Train and Evaluate Logistic Regression (with FE) ---")
lr_model_fe = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
lr_model_fe.fit(X_train_fe, y_train_fe)
y_pred_lr_fe_train = lr_model_fe.predict(X_train_fe)
y_pred_proba_lr_fe_train = lr_model_fe.predict_proba(X_train_fe)
print(f"Accuracy (LR with FE): {accuracy_score(y_train_fe, y_pred_lr_fe_train):.4f}")
roc_auc_lr_fe_ovr = roc_auc_score(y_train_fe, y_pred_proba_lr_fe_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (LR with FE): {roc_auc_lr_fe_ovr:.4f}")

# Initialize Stratified K-Fold Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("\nStratifiedKFold (skf) initialized.")
print("All data preparation and initial model training/evaluation steps re-executed.")

**Reasoning**:
The previous execution indicated `FileNotFoundError` for `train.csv.zip` and `test.csv.zip`. The traceback also showed `NameError: name 'df_train' is not defined` because the data loading failed. I need to re-execute the entire data preparation pipeline to ensure all dataframes and variables are correctly defined. This step directly addresses the error by attempting to re-run the code that loads, cleans, preprocesses, and feature engineers the data, making sure the environment is set up for subsequent modeling steps. I will assume the necessary files are now available in the `/content/` directory.



In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

print("--- Step 1: Loading Datasets ---")
try:
    df_train = pd.read_csv('/content/train.csv.zip', compression='zip')
    print("Train data loaded successfully.")
except FileNotFoundError:
    print("Error: train.csv.zip not found. Please ensure the file is in /content/.")
except Exception as e:
    print(f"An error occurred while loading train data: {e}")

try:
    df_test = pd.read_csv('/content/test.csv.zip', compression='zip')
    print("Test data loaded successfully.")
except FileNotFoundError:
    print("Error: test.csv.zip not found. Please ensure the file is in /content/.")
except Exception as e:
    print(f"An error occurred while loading test data: {e}")


print("\n--- Step 2: Data Cleaning and Type Conversion ---")
def clean_and_convert_data(df):
    # Convert 'Credit_Score' to numerical using Label Encoding (Poor: 0, Standard: 1, Good: 2)
    credit_score_mapping = {'Poor': 0, 'Standard': 1, 'Good': 2}
    if 'Credit_Score' in df.columns:
        df['Credit_Score'] = df['Credit_Score'].map(credit_score_mapping)

    # Clean and convert 'Age' column
    def clean_age(age_val):
        if isinstance(age_val, str):
            age_val = age_val.replace('_', '').replace('+', '')
            if age_val.isdigit():
                age = int(age_val)
                if 0 < age < 100:
                    return age
                elif age < 0:
                    return abs(age)
                else:
                    return np.nan
            else:
                return np.nan
        return age_val

    df['Age'] = df['Age'].apply(clean_age)
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    df.loc[df['Age'] > 100, 'Age'] = np.nan

    # Clean and convert financial object columns to numeric
    financial_cols = [
        'Annual_Income', 'Num_of_Loan', 'Outstanding_Debt',
        'Amount_invested_monthly', 'Monthly_Balance'
    ]
    for col in financial_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(r'[^\\d.]', '', regex=True)
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Clean and convert 'Credit_History_Age' to years
    def convert_credit_history_age(age_str):
        if isinstance(age_str, str):
            parts = age_str.replace('Years', '').replace('Months', '').replace('and', '').split()
            years = 0
            months = 0
            if len(parts) >= 2:
                try:
                    years = int(parts[0])
                except ValueError:
                    pass
                try:
                    months = int(parts[1])
                except ValueError:
                    pass
            return years + months / 12
        return np.nan

    if 'Credit_History_Age' in df.columns:
        df['Credit_History_Age'] = df['Credit_History_Age'].apply(convert_credit_history_age)

    # Handle problematic '-1' in 'Num_Bank_Accounts' and 'Num_of_Loan' by replacing with NaN
    for col in ['Num_Bank_Accounts', 'Num_of_Loan']:
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan

    return df

df_train = clean_and_convert_data(df_train)
df_test = clean_and_convert_data(df_test)
print("Data cleaning and conversion complete.")


print("\n--- Step 3: Handling Missing Values ---")
numeric_cols_to_impute = [
    'Age', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_of_Loan',
    'Num_Credit_Inquiries', 'Credit_History_Age', 'Amount_invested_monthly', 'Monthly_Balance'
]
categorical_cols_to_impute = [
    'Name', 'Type_of_Loan', 'Num_of_Delayed_Payment'
]

for col in numeric_cols_to_impute:
    if col in df_train.columns:
        median_val_train = df_train[col].median()
        df_train[col].fillna(median_val_train, inplace=True)
    if col in df_test.columns:
        median_val_test = df_test[col].median()
        df_test[col].fillna(median_val_test, inplace=True)

for col in categorical_cols_to_impute:
    if col in df_train.columns:
        mode_val_train = df_train[col].mode()[0]
        df_train[col].fillna(mode_val_train, inplace=True)
    if col in df_test.columns:
        mode_val_test = df_test[col].mode()[0]
        df_test[col].fillna(mode_val_test, inplace=True)
print("Missing value imputation complete.")


print("\n--- Step 4: Categorical Feature Encoding ---")
categorical_cols = [col for col in df_train.select_dtypes(include='object').columns
                    if col not in ['ID', 'Customer_ID', 'SSN']]
df_train = pd.get_dummies(df_train, columns=categorical_cols, drop_first=True)
df_test = pd.get_dummies(df_test, columns=categorical_cols, drop_first=True)

train_cols = set(df_train.columns)
test_cols = set(df_test.columns)

missing_in_test = list(train_cols - test_cols)
for col in missing_in_test:
    if col != 'Credit_Score':
        df_test[col] = 0

missing_in_train = list(test_cols - train_cols)
for col in missing_in_train:
    df_train[col] = 0

df_test = df_test[df_train.drop(columns=['Credit_Score'], errors='ignore').columns]
print("Categorical encoding complete.")


print("\n--- Step 5: Feature Engineering ---")
def create_financial_features(df):
    df_copy = df.copy()
    numeric_cols_for_fe = [
        'Annual_Income', 'Monthly_Inhand_Salary', 'Outstanding_Debt',
        'Total_EMI_per_month', 'Monthly_Balance', 'Credit_Utilization_Ratio'
    ]
    for col in numeric_cols_for_fe:
        if col in df_copy.columns:
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')
            if df_copy[col].isnull().any():
                df_copy[col].fillna(df_copy[col].median(), inplace=True)

    df_copy['Annual_Income_adj'] = df_copy['Annual_Income'].replace(0, df_copy['Annual_Income'].median())
    df_copy['Debt_to_Income_Ratio'] = df_copy['Outstanding_Debt'] / df_copy['Annual_Income_adj']
    df_copy.drop('Annual_Income_adj', axis=1, inplace=True)

    df_copy['Monthly_Inhand_Salary_adj'] = df_copy['Monthly_Inhand_Salary'].replace(0, df_copy['Monthly_Inhand_Salary'].median())
    df_copy['Payment_to_Income_Ratio'] = df_copy['Total_EMI_per_month'] / df_copy['Monthly_Inhand_Salary_adj']
    df_copy.drop('Monthly_Inhand_Salary_adj', axis=1, inplace=True)

    df_copy['Annual_Income_adj'] = df_copy['Annual_Income'].replace(0, df_copy['Annual_Income'].median())
    df_copy['Monthly_Income_Annual_Ratio'] = (df_copy['Monthly_Inhand_Salary'] * 12) / df_copy['Annual_Income_adj']
    df_copy.drop('Annual_Income_adj', axis=1, inplace=True)

    df_copy['Monthly_Inhand_Salary_adj'] = df_copy['Monthly_Inhand_Salary'].replace(0, df_copy['Monthly_Inhand_Salary'].median())
    df_copy['Balance_to_Income_Ratio'] = df_copy['Monthly_Balance'] / df_copy['Monthly_Inhand_Salary_adj']
    df_copy.drop('Monthly_Inhand_Salary_adj', axis=1, inplace=True)

    for col in ['Debt_to_Income_Ratio', 'Payment_to_Income_Ratio', 'Monthly_Income_Annual_Ratio', 'Balance_to_Income_Ratio']:
        if col in df_copy.columns:
            df_copy[col].fillna(df_copy[col].median(), inplace=True)

    return df_copy

df_train_fe = create_financial_features(df_train)
df_test_fe = create_financial_features(df_test)
print("Feature engineering complete. DataFrames: df_train_fe, df_test_fe created.")


print("\n--- Step 6: Prepare Data for Modeling (Pre-Feature Engineering) ---")
non_feature_cols = ['ID', 'Customer_ID', 'SSN']
X_train = df_train.drop(columns=['Credit_Score'] + [col for col in non_feature_cols if col in df_train.columns], errors='ignore')
y_train = df_train['Credit_Score']
X_test = df_test.drop(columns=[col for col in non_feature_cols if col in df_test.columns], errors='ignore')

# Align columns for X_test (pre-FE)
missing_cols_pre_fe = set(X_train.columns) - set(X_test.columns)
for c in missing_cols_pre_fe:
    X_test[c] = 0
X_test = X_test[X_train.columns]

print("X_train, y_train, X_test (pre-FE) prepared.")


print("\n--- Step 7: Train and Evaluate Logistic Regression (Original) ---")
model = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
model.fit(X_train, y_train)
y_pred_train = model.predict(X_train)
y_pred_proba_train = model.predict_proba(X_train)
print(f"Accuracy (LR): {accuracy_score(y_train, y_pred_train):.4f}")
roc_auc_ovr = roc_auc_score(y_train, y_pred_proba_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (LR): {roc_auc_ovr:.4f}")


print("\n--- Step 8: Train and Evaluate Random Forest ---")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)
y_pred_rf_train = rf_model.predict(X_train)
y_pred_proba_rf_train = rf_model.predict_proba(X_train)
print(f"Accuracy (RF): {accuracy_score(y_train, y_pred_rf_train):.4f}")
roc_auc_rf_ovr = roc_auc_score(y_train, y_pred_proba_rf_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (RF): {roc_auc_rf_ovr:.4f}")


print("\n--- Step 9: Train and Evaluate Decision Tree ---")
dt_model = DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10)
dt_model.fit(X_train, y_train)
y_pred_dt_train = dt_model.predict(X_train)
y_pred_proba_dt_train = dt_model.predict_proba(X_train)
print(f"Accuracy (DT): {accuracy_score(y_train, y_pred_dt_train):.4f}")
roc_auc_dt_ovr = roc_auc_score(y_train, y_pred_proba_dt_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (DT): {roc_auc_ovr:.4f}")


print("\n--- Step 10: Prepare Data for Modeling (Post-Feature Engineering) ---")
non_feature_cols = ['ID', 'Customer_ID', 'SSN']
X_train_fe = df_train_fe.drop(columns=['Credit_Score'] + [col for col in non_feature_cols if col in df_train_fe.columns], errors='ignore')
y_train_fe = df_train_fe['Credit_Score']
X_test_fe = df_test_fe.drop(columns=[col for col in non_feature_cols if col in df_test_fe.columns], errors='ignore')

# Align columns for X_test_fe (post-FE)
train_fe_cols = set(X_train_fe.columns)
test_fe_cols = set(X_test_fe.columns)

missing_in_test_fe = list(train_fe_cols - test_fe_cols)
for col in missing_in_test_fe:
    X_test_fe[col] = 0

missing_in_train_fe = list(test_fe_cols - train_fe_cols)
for col in missing_in_train_fe:
    X_train_fe[col] = 0

X_test_fe = X_test_fe[X_train_fe.columns]
print("X_train_fe, y_train_fe, X_test_fe (post-FE) prepared.")


print("\n--- Step 11: Train and Evaluate Logistic Regression (with FE) ---")
lr_model_fe = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
lr_model_fe.fit(X_train_fe, y_train_fe)
y_pred_lr_fe_train = lr_model_fe.predict(X_train_fe)
y_pred_proba_lr_fe_train = lr_model_fe.predict_proba(X_train_fe)
print(f"Accuracy (LR with FE): {accuracy_score(y_train_fe, y_pred_lr_fe_train):.4f}")
roc_auc_lr_fe_ovr = roc_auc_score(y_train_fe, y_pred_proba_lr_fe_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (LR with FE): {roc_auc_lr_fe_ovr:.4f}")

# Initialize Stratified K-Fold Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("\nStratifiedKFold (skf) initialized.")
print("All data preparation and initial model training/evaluation steps re-executed.")

--- Step 1: Loading Datasets ---
Error: train.csv.zip not found. Please ensure the file is in /content/.
Error: test.csv.zip not found. Please ensure the file is in /content/.

--- Step 2: Data Cleaning and Type Conversion ---


NameError: name 'df_train' is not defined

**Reasoning**:
Re-executing the entire data preparation, cleaning, feature engineering, and model initialization pipeline to ensure all necessary dataframes and variables, including `X_train_fe`, `y_train_fe`, and `skf`, are correctly defined and ready for cross-validation.



In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

print("--- Step 1: Loading Datasets ---")
try:
    df_train = pd.read_csv('/content/train.csv.zip', compression='zip')
    print("Train data loaded successfully.")
except FileNotFoundError:
    print("Error: train.csv.zip not found. Please ensure the file is in /content/.")
except Exception as e:
    print(f"An error occurred while loading train data: {e}")

try:
    df_test = pd.read_csv('/content/test.csv.zip', compression='zip')
    print("Test data loaded successfully.")
except FileNotFoundError:
    print("Error: test.csv.zip not found. Please ensure the file is in /content/.")
except Exception as e:
    print(f"An error occurred while loading test data: {e}")


print("\n--- Step 2: Data Cleaning and Type Conversion ---")
def clean_and_convert_data(df):
    # Convert 'Credit_Score' to numerical using Label Encoding (Poor: 0, Standard: 1, Good: 2)
    credit_score_mapping = {'Poor': 0, 'Standard': 1, 'Good': 2}
    if 'Credit_Score' in df.columns:
        df['Credit_Score'] = df['Credit_Score'].map(credit_score_mapping)

    # Clean and convert 'Age' column
    def clean_age(age_val):
        if isinstance(age_val, str):
            age_val = age_val.replace('_', '').replace('+', '')
            if age_val.isdigit():
                age = int(age_val)
                if 0 < age < 100:
                    return age
                elif age < 0:
                    return abs(age)
                else:
                    return np.nan
            else:
                return np.nan
        return age_val

    df['Age'] = df['Age'].apply(clean_age)
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    df.loc[df['Age'] > 100, 'Age'] = np.nan

    # Clean and convert financial object columns to numeric
    financial_cols = [
        'Annual_Income', 'Num_of_Loan', 'Outstanding_Debt',
        'Amount_invested_monthly', 'Monthly_Balance'
    ]
    for col in financial_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(r'[^\\d.]', '', regex=True)
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Clean and convert 'Credit_History_Age' to years
    def convert_credit_history_age(age_str):
        if isinstance(age_str, str):
            parts = age_str.replace('Years', '').replace('Months', '').replace('and', '').split()
            years = 0
            months = 0
            if len(parts) >= 2:
                try:
                    years = int(parts[0])
                except ValueError:
                    pass
                try:
                    months = int(parts[1])
                except ValueError:
                    pass
            return years + months / 12
        return np.nan

    if 'Credit_History_Age' in df.columns:
        df['Credit_History_Age'] = df['Credit_History_Age'].apply(convert_credit_history_age)

    # Handle problematic '-1' in 'Num_Bank_Accounts' and 'Num_of_Loan' by replacing with NaN
    for col in ['Num_Bank_Accounts', 'Num_of_Loan']:
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan

    return df

df_train = clean_and_convert_data(df_train)
df_test = clean_and_convert_data(df_test)
print("Data cleaning and conversion complete.")


print("\n--- Step 3: Handling Missing Values ---")
numeric_cols_to_impute = [
    'Age', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_of_Loan',
    'Num_Credit_Inquiries', 'Credit_History_Age', 'Amount_invested_monthly', 'Monthly_Balance'
]
categorical_cols_to_impute = [
    'Name', 'Type_of_Loan', 'Num_of_Delayed_Payment'
]

for col in numeric_cols_to_impute:
    if col in df_train.columns:
        median_val_train = df_train[col].median()
        df_train[col].fillna(median_val_train, inplace=True)
    if col in df_test.columns:
        median_val_test = df_test[col].median()
        df_test[col].fillna(median_val_test, inplace=True)

for col in categorical_cols_to_impute:
    if col in df_train.columns:
        mode_val_train = df_train[col].mode()[0]
        df_train[col].fillna(mode_val_train, inplace=True)
    if col in df_test.columns:
        mode_val_test = df_test[col].mode()[0]
        df_test[col].fillna(mode_val_test, inplace=True)
print("Missing value imputation complete.")


print("\n--- Step 4: Categorical Feature Encoding ---")
categorical_cols = [col for col in df_train.select_dtypes(include='object').columns
                    if col not in ['ID', 'Customer_ID', 'SSN']]
df_train = pd.get_dummies(df_train, columns=categorical_cols, drop_first=True)
df_test = pd.get_dummies(df_test, columns=categorical_cols, drop_first=True)

train_cols = set(df_train.columns)
test_cols = set(df_test.columns)

missing_in_test = list(train_cols - test_cols)
for col in missing_in_test:
    if col != 'Credit_Score':
        df_test[col] = 0

missing_in_train = list(test_cols - train_cols)
for col in missing_in_train:
    df_train[col] = 0

df_test = df_test[df_train.drop(columns=['Credit_Score'], errors='ignore').columns]
print("Categorical encoding complete.")


print("\n--- Step 5: Feature Engineering ---")
def create_financial_features(df):
    df_copy = df.copy()
    numeric_cols_for_fe = [
        'Annual_Income', 'Monthly_Inhand_Salary', 'Outstanding_Debt',
        'Total_EMI_per_month', 'Monthly_Balance', 'Credit_Utilization_Ratio'
    ]
    for col in numeric_cols_for_fe:
        if col in df_copy.columns:
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')
            if df_copy[col].isnull().any():
                df_copy[col].fillna(df_copy[col].median(), inplace=True)

    df_copy['Annual_Income_adj'] = df_copy['Annual_Income'].replace(0, df_copy['Annual_Income'].median())
    df_copy['Debt_to_Income_Ratio'] = df_copy['Outstanding_Debt'] / df_copy['Annual_Income_adj']
    df_copy.drop('Annual_Income_adj', axis=1, inplace=True)

    df_copy['Monthly_Inhand_Salary_adj'] = df_copy['Monthly_Inhand_Salary'].replace(0, df_copy['Monthly_Inhand_Salary'].median())
    df_copy['Payment_to_Income_Ratio'] = df_copy['Total_EMI_per_month'] / df_copy['Monthly_Inhand_Salary_adj']
    df_copy.drop('Monthly_Inhand_Salary_adj', axis=1, inplace=True)

    df_copy['Annual_Income_adj'] = df_copy['Annual_Income'].replace(0, df_copy['Annual_Income'].median())
    df_copy['Monthly_Income_Annual_Ratio'] = (df_copy['Monthly_Inhand_Salary'] * 12) / df_copy['Annual_Income_adj']
    df_copy.drop('Annual_Income_adj', axis=1, inplace=True)

    df_copy['Monthly_Inhand_Salary_adj'] = df_copy['Monthly_Inhand_Salary'].replace(0, df_copy['Monthly_Inhand_Salary'].median())
    df_copy['Balance_to_Income_Ratio'] = df_copy['Monthly_Balance'] / df_copy['Monthly_Inhand_Salary_adj']
    df_copy.drop('Monthly_Inhand_Salary_adj', axis=1, inplace=True)

    for col in ['Debt_to_Income_Ratio', 'Payment_to_Income_Ratio', 'Monthly_Income_Annual_Ratio', 'Balance_to_Income_Ratio']:
        if col in df_copy.columns:
            df_copy[col].fillna(df_copy[col].median(), inplace=True)

    return df_copy

df_train_fe = create_financial_features(df_train)
df_test_fe = create_financial_features(df_test)
print("Feature engineering complete. DataFrames: df_train_fe, df_test_fe created.")


print("\n--- Step 6: Prepare Data for Modeling (Pre-Feature Engineering) ---")
non_feature_cols = ['ID', 'Customer_ID', 'SSN']
X_train = df_train.drop(columns=['Credit_Score'] + [col for col in non_feature_cols if col in df_train.columns], errors='ignore')
y_train = df_train['Credit_Score']
X_test = df_test.drop(columns=[col for col in non_feature_cols if col in df_test.columns], errors='ignore')

# Align columns for X_test (pre-FE)
missing_cols_pre_fe = set(X_train.columns) - set(X_test.columns)
for c in missing_cols_pre_fe:
    X_test[c] = 0
X_test = X_test[X_train.columns]

print("X_train, y_train, X_test (pre-FE) prepared.")


print("\n--- Step 7: Train and Evaluate Logistic Regression (Original) ---")
model = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
model.fit(X_train, y_train)
y_pred_train = model.predict(X_train)
y_pred_proba_train = model.predict_proba(X_train)
print(f"Accuracy (LR): {accuracy_score(y_train, y_pred_train):.4f}")
roc_auc_ovr = roc_auc_score(y_train, y_pred_proba_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (LR): {roc_auc_ovr:.4f}")


print("\n--- Step 8: Train and Evaluate Random Forest ---")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)
y_pred_rf_train = rf_model.predict(X_train)
y_pred_proba_rf_train = rf_model.predict_proba(X_train)
print(f"Accuracy (RF): {accuracy_score(y_train, y_pred_rf_train):.4f}")
roc_auc_rf_ovr = roc_auc_score(y_train, y_pred_proba_rf_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (RF): {roc_auc_rf_ovr:.4f}")


print("\n--- Step 9: Train and Evaluate Decision Tree ---")
dt_model = DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10)
dt_model.fit(X_train, y_train)
y_pred_dt_train = dt_model.predict(X_train)
y_pred_proba_dt_train = dt_model.predict_proba(X_train)
print(f"Accuracy (DT): {accuracy_score(y_train, y_pred_dt_train):.4f}")
roc_auc_dt_ovr = roc_auc_score(y_train, y_pred_proba_dt_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (DT): {roc_auc_ovr:.4f}")


print("\n--- Step 10: Prepare Data for Modeling (Post-Feature Engineering) ---")
non_feature_cols = ['ID', 'Customer_ID', 'SSN']
X_train_fe = df_train_fe.drop(columns=['Credit_Score'] + [col for col in non_feature_cols if col in df_train_fe.columns], errors='ignore')
y_train_fe = df_train_fe['Credit_Score']
X_test_fe = df_test_fe.drop(columns=[col for col in non_feature_cols if col in df_test_fe.columns], errors='ignore')

# Align columns for X_test_fe (post-FE)
train_fe_cols = set(X_train_fe.columns)
test_fe_cols = set(X_test_fe.columns)

missing_in_test_fe = list(train_fe_cols - test_fe_cols)
for col in missing_in_test_fe:
    X_test_fe[col] = 0

missing_in_train_fe = list(test_fe_cols - train_fe_cols)
for col in missing_in_train_fe:
    X_train_fe[col] = 0

X_test_fe = X_test_fe[X_train_fe.columns]
print("X_train_fe, y_train_fe, X_test_fe (post-FE) prepared.")


print("\n--- Step 11: Train and Evaluate Logistic Regression (with FE) ---")
lr_model_fe = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
lr_model_fe.fit(X_train_fe, y_train_fe)
y_pred_lr_fe_train = lr_model_fe.predict(X_train_fe)
y_pred_proba_lr_fe_train = lr_model_fe.predict_proba(X_train_fe)
print(f"Accuracy (LR with FE): {accuracy_score(y_train_fe, y_pred_lr_fe_train):.4f}")
roc_auc_lr_fe_ovr = roc_auc_score(y_train_fe, y_pred_proba_lr_fe_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (LR with FE): {roc_auc_lr_fe_ovr:.4f}")

# Initialize Stratified K-Fold Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("\nStratifiedKFold (skf) initialized.")
print("All data preparation and initial model training/evaluation steps re-executed.")

It appears the `train.csv.zip` and `test.csv.zip` files are still not found in the `/content/` directory, causing the `NameError` for `df_train`.

**Please upload both `train.csv.zip` and `test.csv.zip` files to the `/content/` directory.** You can do this by clicking the folder icon on the left sidebar, navigating to `/content/`, and then dragging and dropping the files there.

Once you have uploaded the files, please let me know, and I will re-execute the data preparation and feature engineering step.

## Perform Cross-Validation for Models

### Subtask:
Implement 5-fold stratified cross-validation to evaluate Logistic Regression, Random Forest, and Decision Tree models on the training data with feature engineered features (`X_train_fe`, `y_train_fe`) using multiple scoring metrics, including accuracy and ROC-AUC.


In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

print("--- Step 1: Loading Datasets ---")
try:
    df_train = pd.read_csv('/content/train.csv.zip', compression='zip')
    print("Train data loaded successfully.")
except FileNotFoundError:
    print("Error: train.csv.zip not found. Please ensure the file is in /content/.")
except Exception as e:
    print(f"An error occurred while loading train data: {e}")

try:
    df_test = pd.read_csv('/content/test.csv.zip', compression='zip')
    print("Test data loaded successfully.")
except FileNotFoundError:
    print("Error: test.csv.zip not found. Please ensure the file is in /content/.")
except Exception as e:
    print(f"An error occurred while loading test data: {e}")


print("\n--- Step 2: Data Cleaning and Type Conversion ---")
def clean_and_convert_data(df):
    # Convert 'Credit_Score' to numerical using Label Encoding (Poor: 0, Standard: 1, Good: 2)
    credit_score_mapping = {'Poor': 0, 'Standard': 1, 'Good': 2}
    if 'Credit_Score' in df.columns:
        df['Credit_Score'] = df['Credit_Score'].map(credit_score_mapping)

    # Clean and convert 'Age' column
    def clean_age(age_val):
        if isinstance(age_val, str):
            age_val = age_val.replace('_', '').replace('+', '')
            if age_val.isdigit():
                age = int(age_val)
                if 0 < age < 100:
                    return age
                elif age < 0:
                    return abs(age)
                else:
                    return np.nan
            else:
                return np.nan
        return age_val

    df['Age'] = df['Age'].apply(clean_age)
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    df.loc[df['Age'] > 100, 'Age'] = np.nan

    # Clean and convert financial object columns to numeric
    financial_cols = [
        'Annual_Income', 'Num_of_Loan', 'Outstanding_Debt',
        'Amount_invested_monthly', 'Monthly_Balance'
    ]
    for col in financial_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(r'[^\\d.]', '', regex=True)
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Clean and convert 'Credit_History_Age' to years
    def convert_credit_history_age(age_str):
        if isinstance(age_str, str):
            parts = age_str.replace('Years', '').replace('Months', '').replace('and', '').split()
            years = 0
            months = 0
            if len(parts) >= 2:
                try:
                    years = int(parts[0])
                except ValueError:
                    pass
                try:
                    months = int(parts[1])
                except ValueError:
                    pass
            return years + months / 12
        return np.nan

    if 'Credit_History_Age' in df.columns:
        df['Credit_History_Age'] = df['Credit_History_Age'].apply(convert_credit_history_age)

    # Handle problematic '-1' in 'Num_Bank_Accounts' and 'Num_of_Loan' by replacing with NaN
    for col in ['Num_Bank_Accounts', 'Num_of_Loan']:
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan

    return df

df_train = clean_and_convert_data(df_train)
df_test = clean_and_convert_data(df_test)
print("Data cleaning and conversion complete.")


print("\n--- Step 3: Handling Missing Values ---")
numeric_cols_to_impute = [
    'Age', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_of_Loan',
    'Num_Credit_Inquiries', 'Credit_History_Age', 'Amount_invested_monthly', 'Monthly_Balance'
]
categorical_cols_to_impute = [
    'Name', 'Type_of_Loan', 'Num_of_Delayed_Payment'
]

for col in numeric_cols_to_impute:
    if col in df_train.columns:
        median_val_train = df_train[col].median()
        df_train[col].fillna(median_val_train, inplace=True)
    if col in df_test.columns:
        median_val_test = df_test[col].median()
        df_test[col].fillna(median_val_test, inplace=True)

for col in categorical_cols_to_impute:
    if col in df_train.columns:
        mode_val_train = df_train[col].mode()[0]
        df_train[col].fillna(mode_val_train, inplace=True)
    if col in df_test.columns:
        mode_val_test = df_test[col].mode()[0]
        df_test[col].fillna(mode_val_test, inplace=True)
print("Missing value imputation complete.")


print("\n--- Step 4: Categorical Feature Encoding ---")
categorical_cols = [col for col in df_train.select_dtypes(include='object').columns
                    if col not in ['ID', 'Customer_ID', 'SSN']]
df_train = pd.get_dummies(df_train, columns=categorical_cols, drop_first=True)
df_test = pd.get_dummies(df_test, columns=categorical_cols, drop_first=True)

train_cols = set(df_train.columns)
test_cols = set(df_test.columns)

missing_in_test = list(train_cols - test_cols)
for col in missing_in_test:
    if col != 'Credit_Score':
        df_test[col] = 0

missing_in_train = list(test_cols - train_cols)
for col in missing_in_train:
    df_train[col] = 0

df_test = df_test[df_train.drop(columns=['Credit_Score'], errors='ignore').columns]
print("Categorical encoding complete.")


print("\n--- Step 5: Feature Engineering ---")
def create_financial_features(df):
    df_copy = df.copy()
    numeric_cols_for_fe = [
        'Annual_Income', 'Monthly_Inhand_Salary', 'Outstanding_Debt',
        'Total_EMI_per_month', 'Monthly_Balance', 'Credit_Utilization_Ratio'
    ]
    for col in numeric_cols_for_fe:
        if col in df_copy.columns:
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')
            if df_copy[col].isnull().any():
                df_copy[col].fillna(df_copy[col].median(), inplace=True)

    df_copy['Annual_Income_adj'] = df_copy['Annual_Income'].replace(0, df_copy['Annual_Income'].median())
    df_copy['Debt_to_Income_Ratio'] = df_copy['Outstanding_Debt'] / df_copy['Annual_Income_adj']
    df_copy.drop('Annual_Income_adj', axis=1, inplace=True)

    df_copy['Monthly_Inhand_Salary_adj'] = df_copy['Monthly_Inhand_Salary'].replace(0, df_copy['Monthly_Inhand_Salary'].median())
    df_copy['Payment_to_Income_Ratio'] = df_copy['Total_EMI_per_month'] / df_copy['Monthly_Inhand_Salary_adj']
    df_copy.drop('Monthly_Inhand_Salary_adj', axis=1, inplace=True)

    df_copy['Annual_Income_adj'] = df_copy['Annual_Income'].replace(0, df_copy['Annual_Income'].median())
    df_copy['Monthly_Income_Annual_Ratio'] = (df_copy['Monthly_Inhand_Salary'] * 12) / df_copy['Annual_Income_adj']
    df_copy.drop('Annual_Income_adj', axis=1, inplace=True)

    df_copy['Monthly_Inhand_Salary_adj'] = df_copy['Monthly_Inhand_Salary'].replace(0, df_copy['Monthly_Inhand_Salary'].median())
    df_copy['Balance_to_Income_Ratio'] = df_copy['Monthly_Balance'] / df_copy['Monthly_Inhand_Salary_adj']
    df_copy.drop('Monthly_Inhand_Salary_adj', axis=1, inplace=True)

    for col in ['Debt_to_Income_Ratio', 'Payment_to_Income_Ratio', 'Monthly_Income_Annual_Ratio', 'Balance_to_Income_Ratio']:
        if col in df_copy.columns:
            df_copy[col].fillna(df_copy[col].median(), inplace=True)

    return df_copy

df_train_fe = create_financial_features(df_train)
df_test_fe = create_financial_features(df_test)
print("Feature engineering complete. DataFrames: df_train_fe, df_test_fe created.")


print("\n--- Step 6: Prepare Data for Modeling (Pre-Feature Engineering) ---")
non_feature_cols = ['ID', 'Customer_ID', 'SSN']
X_train = df_train.drop(columns=['Credit_Score'] + [col for col in non_feature_cols if col in df_train.columns], errors='ignore')
y_train = df_train['Credit_Score']
X_test = df_test.drop(columns=[col for col in non_feature_cols if col in df_test.columns], errors='ignore')

# Align columns for X_test (pre-FE)
missing_cols_pre_fe = set(X_train.columns) - set(X_test.columns)
for c in missing_cols_pre_fe:
    X_test[c] = 0
X_test = X_test[X_train.columns]

print("X_train, y_train, X_test (pre-FE) prepared.")


print("\n--- Step 7: Train and Evaluate Logistic Regression (Original) ---")
model = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
model.fit(X_train, y_train)
y_pred_train = model.predict(X_train)
y_pred_proba_train = model.predict_proba(X_train)
print(f"Accuracy (LR): {accuracy_score(y_train, y_pred_train):.4f}")
roc_auc_ovr = roc_auc_score(y_train, y_pred_proba_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (LR): {roc_auc_ovr:.4f}")


print("\n--- Step 8: Train and Evaluate Random Forest ---")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)
y_pred_rf_train = rf_model.predict(X_train)
y_pred_proba_rf_train = rf_model.predict_proba(X_train)
print(f"Accuracy (RF): {accuracy_score(y_train, y_pred_rf_train):.4f}")
roc_auc_rf_ovr = roc_auc_score(y_train, y_pred_proba_rf_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (RF): {roc_auc_rf_ovr:.4f}")


print("\n--- Step 9: Train and Evaluate Decision Tree ---")
dt_model = DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10)
dt_model.fit(X_train, y_train)
y_pred_dt_train = dt_model.predict(X_train)
y_pred_proba_dt_train = dt_model.predict_proba(X_train)
print(f"Accuracy (DT): {accuracy_score(y_train, y_pred_dt_train):.4f}")
roc_auc_dt_ovr = roc_auc_score(y_train, y_pred_proba_dt_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (DT): {roc_auc_ovr:.4f}")


print("\n--- Step 10: Prepare Data for Modeling (Post-Feature Engineering) ---")
non_feature_cols = ['ID', 'Customer_ID', 'SSN']
X_train_fe = df_train_fe.drop(columns=['Credit_Score'] + [col for col in non_feature_cols if col in df_train_fe.columns], errors='ignore')
y_train_fe = df_train_fe['Credit_Score']
X_test_fe = df_test_fe.drop(columns=[col for col in non_feature_cols if col in df_test_fe.columns], errors='ignore')

# Align columns for X_test_fe (post-FE)
train_fe_cols = set(X_train_fe.columns)
test_fe_cols = set(X_test_fe.columns)

missing_in_test_fe = list(train_fe_cols - test_fe_cols)
for col in missing_in_test_fe:
    X_test_fe[col] = 0

missing_in_train_fe = list(test_fe_cols - train_fe_cols)
for col in missing_in_train_fe:
    X_train_fe[col] = 0

X_test_fe = X_test_fe[X_train_fe.columns]
print("X_train_fe, y_train_fe, X_test_fe (post-FE) prepared.")


print("\n--- Step 11: Train and Evaluate Logistic Regression (with FE) ---")
lr_model_fe = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
lr_model_fe.fit(X_train_fe, y_train_fe)
y_pred_lr_fe_train = lr_model_fe.predict(X_train_fe)
y_pred_proba_lr_fe_train = lr_model_fe.predict_proba(X_train_fe)
print(f"Accuracy (LR with FE): {accuracy_score(y_train_fe, y_pred_lr_fe_train):.4f}")
roc_auc_lr_fe_ovr = roc_auc_score(y_train_fe, y_pred_proba_lr_fe_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (LR with FE): {roc_auc_lr_fe_ovr:.4f}")

# Initialize Stratified K-Fold Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("\nStratifiedKFold (skf) initialized.")
print("All data preparation and initial model training/evaluation steps re-executed.")

# Define the models
models = {
    "Logistic Regression": LogisticRegression(solver='liblinear', random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10)
}

# Define scoring metrics
scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc_ovr']

results = {}

print("--- Starting Cross-Validation ---")

for name, model in models.items():
    print(f"\nEvaluating {name}...")
    cv_results = cross_validate(model, X_train_fe, y_train_fe, cv=skf, scoring=scoring, n_jobs=-1, verbose=0)

    results[name] = {
        'Accuracy': np.mean(cv_results['test_accuracy']),
        'Precision': np.mean(cv_results['test_precision_macro']),
        'Recall': np.mean(cv_results['test_recall_macro']),
        'F1-Score': np.mean(cv_results['test_f1_macro']),
        'ROC-AUC': np.mean(cv_results['test_roc_auc_ovr'])
    }

    print(f"{name} - Mean Accuracy: {results[name]['Accuracy']:.4f} (+/- {np.std(cv_results['test_accuracy']):.4f})")
    print(f"{name} - Mean Precision: {results[name]['Precision']:.4f} (+/- {np.std(cv_results['test_precision_macro']):.4f})")
    print(f"{name} - Mean Recall: {results[name]['Recall']:.4f} (+/- {np.std(cv_results['test_recall_macro']):.4f})")
    print(f"{name} - Mean F1-Score: {results[name]['F1-Score']:.4f} (+/- {np.std(cv_results['test_f1_macro']):.4f})")
    print(f"{name} - Mean ROC-AUC: {results[name]['ROC-AUC']:.4f} (+/- {np.std(cv_results['test_roc_auc_ovr']):.4f})")

print("\n--- Cross-Validation Complete ---")

# Display all results in a DataFrame for easier comparison
results_df = pd.DataFrame(results).T
print("\nSummary of Cross-Validation Results:")
display(results_df)

--- Step 1: Loading Datasets ---
Error: train.csv.zip not found. Please ensure the file is in /content/.
Error: test.csv.zip not found. Please ensure the file is in /content/.

--- Step 2: Data Cleaning and Type Conversion ---


NameError: name 'df_train' is not defined

#### Re-executing Data Preparation

With the `train.csv.zip` and `test.csv.zip` files now available in the `/content/` directory, I will re-run the data loading, cleaning, missing value imputation, categorical encoding, and feature engineering steps to correctly define all necessary dataframes and feature sets.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

print("--- Step 1: Loading Datasets ---")
try:
    df_train = pd.read_csv('/content/train.csv.zip', compression='zip')
    print("Train data loaded successfully.")
except FileNotFoundError:
    print("Error: train.csv.zip not found. Please ensure the file is in /content/.")
except Exception as e:
    print(f"An error occurred while loading train data: {e}")

try:
    df_test = pd.read_csv('/content/test.csv.zip', compression='zip')
    print("Test data loaded successfully.")
except FileNotFoundError:
    print("Error: test.csv.zip not found. Please ensure the file is in /content/.")
except Exception as e:
    print(f"An error occurred while loading test data: {e}")


print("\n--- Step 2: Data Cleaning and Type Conversion ---")
def clean_and_convert_data(df):
    # Convert 'Credit_Score' to numerical using Label Encoding (Poor: 0, Standard: 1, Good: 2)
    credit_score_mapping = {'Poor': 0, 'Standard': 1, 'Good': 2}
    if 'Credit_Score' in df.columns:
        df['Credit_Score'] = df['Credit_Score'].map(credit_score_mapping)

    # Clean and convert 'Age' column
    def clean_age(age_val):
        if isinstance(age_val, str):
            age_val = age_val.replace('_', '').replace('+', '')
            if age_val.isdigit():
                age = int(age_val)
                if 0 < age < 100:
                    return age
                elif age < 0:
                    return abs(age)
                else:
                    return np.nan
            else:
                return np.nan
        return age_val

    df['Age'] = df['Age'].apply(clean_age)
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    df.loc[df['Age'] > 100, 'Age'] = np.nan

    # Clean and convert financial object columns to numeric
    financial_cols = [
        'Annual_Income', 'Num_of_Loan', 'Outstanding_Debt',
        'Amount_invested_monthly', 'Monthly_Balance'
    ]
    for col in financial_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(r'[^\\d.]', '', regex=True)
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Clean and convert 'Credit_History_Age' to years
    def convert_credit_history_age(age_str):
        if isinstance(age_str, str):
            parts = age_str.replace('Years', '').replace('Months', '').replace('and', '').split()
            years = 0
            months = 0
            if len(parts) >= 2:
                try:
                    years = int(parts[0])
                except ValueError:
                    pass
                try:
                    months = int(parts[1])
                except ValueError:
                    pass
            return years + months / 12
        return np.nan

    if 'Credit_History_Age' in df.columns:
        df['Credit_History_Age'] = df['Credit_History_Age'].apply(convert_credit_history_age)

    # Handle problematic '-1' in 'Num_Bank_Accounts' and 'Num_of_Loan' by replacing with NaN
    for col in ['Num_Bank_Accounts', 'Num_of_Loan']:
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan

    return df

df_train = clean_and_convert_data(df_train)
df_test = clean_and_convert_data(df_test)
print("Data cleaning and conversion complete.")


print("\n--- Step 3: Handling Missing Values ---")
numeric_cols_to_impute = [
    'Age', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_of_Loan',
    'Num_Credit_Inquiries', 'Credit_History_Age', 'Amount_invested_monthly', 'Monthly_Balance'
]
categorical_cols_to_impute = [
    'Name', 'Type_of_Loan', 'Num_of_Delayed_Payment'
]

for col in numeric_cols_to_impute:
    if col in df_train.columns:
        median_val_train = df_train[col].median()
        df_train[col].fillna(median_val_train, inplace=True)
    if col in df_test.columns:
        median_val_test = df_test[col].median()
        df_test[col].fillna(median_val_test, inplace=True)

for col in categorical_cols_to_impute:
    if col in df_train.columns:
        mode_val_train = df_train[col].mode()[0]
        df_train[col].fillna(mode_val_train, inplace=True)
    if col in df_test.columns:
        mode_val_test = df_test[col].mode()[0]
        df_test[col].fillna(mode_val_test, inplace=True)
print("Missing value imputation complete.")


print("\n--- Step 4: Categorical Feature Encoding ---")
categorical_cols = [col for col in df_train.select_dtypes(include='object').columns
                    if col not in ['ID', 'Customer_ID', 'SSN']]
df_train = pd.get_dummies(df_train, columns=categorical_cols, drop_first=True)
df_test = pd.get_dummies(df_test, columns=categorical_cols, drop_first=True)

train_cols = set(df_train.columns)
test_cols = set(df_test.columns)

missing_in_test = list(train_cols - test_cols)
for col in missing_in_test:
    if col != 'Credit_Score':
        df_test[col] = 0

missing_in_train = list(test_cols - train_cols)
for col in missing_in_train:
    df_train[col] = 0

df_test = df_test[df_train.drop(columns=['Credit_Score'], errors='ignore').columns]
print("Categorical encoding complete.")


print("\n--- Step 5: Feature Engineering ---")
def create_financial_features(df):
    df_copy = df.copy()
    numeric_cols_for_fe = [
        'Annual_Income', 'Monthly_Inhand_Salary', 'Outstanding_Debt',
        'Total_EMI_per_month', 'Monthly_Balance', 'Credit_Utilization_Ratio'
    ]
    for col in numeric_cols_for_fe:
        if col in df_copy.columns:
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')
            if df_copy[col].isnull().any():
                df_copy[col].fillna(df_copy[col].median(), inplace=True)

    df_copy['Annual_Income_adj'] = df_copy['Annual_Income'].replace(0, df_copy['Annual_Income'].median())
    df_copy['Debt_to_Income_Ratio'] = df_copy['Outstanding_Debt'] / df_copy['Annual_Income_adj']
    df_copy.drop('Annual_Income_adj', axis=1, inplace=True)

    df_copy['Monthly_Inhand_Salary_adj'] = df_copy['Monthly_Inhand_Salary'].replace(0, df_copy['Monthly_Inhand_Salary'].median())
    df_copy['Payment_to_Income_Ratio'] = df_copy['Total_EMI_per_month'] / df_copy['Monthly_Inhand_Salary_adj']
    df_copy.drop('Monthly_Inhand_Salary_adj', axis=1, inplace=True)

    df_copy['Annual_Income_adj'] = df_copy['Annual_Income'].replace(0, df_copy['Annual_Income'].median())
    df_copy['Monthly_Income_Annual_Ratio'] = (df_copy['Monthly_Inhand_Salary'] * 12) / df_copy['Annual_Income_adj']
    df_copy.drop('Annual_Income_adj', axis=1, inplace=True)

    df_copy['Monthly_Inhand_Salary_adj'] = df_copy['Monthly_Inhand_Salary'].replace(0, df_copy['Monthly_Inhand_Salary'].median())
    df_copy['Balance_to_Income_Ratio'] = df_copy['Monthly_Balance'] / df_copy['Monthly_Inhand_Salary_adj']
    df_copy.drop('Monthly_Inhand_Salary_adj', axis=1, inplace=True)

    for col in ['Debt_to_Income_Ratio', 'Payment_to_Income_Ratio', 'Monthly_Income_Annual_Ratio', 'Balance_to_Income_Ratio']:
        if col in df_copy.columns:
            df_copy[col].fillna(df_copy[col].median(), inplace=True)

    return df_copy

df_train_fe = create_financial_features(df_train)
df_test_fe = create_financial_features(df_test)
print("Feature engineering complete. DataFrames: df_train_fe, df_test_fe created.")


print("\n--- Step 6: Prepare Data for Modeling (Pre-Feature Engineering) ---")
non_feature_cols = ['ID', 'Customer_ID', 'SSN']
X_train = df_train.drop(columns=['Credit_Score'] + [col for col in non_feature_cols if col in df_train.columns], errors='ignore')
y_train = df_train['Credit_Score']
X_test = df_test.drop(columns=[col for col in non_feature_cols if col in df_test.columns], errors='ignore')

# Align columns for X_test (pre-FE)
missing_cols_pre_fe = set(X_train.columns) - set(X_test.columns)
for c in missing_cols_pre_fe:
    X_test[c] = 0
X_test = X_test[X_train.columns]

print("X_train, y_train, X_test (pre-FE) prepared.")


print("\n--- Step 7: Train and Evaluate Logistic Regression (Original) ---")
model = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
model.fit(X_train, y_train)
y_pred_train = model.predict(X_train)
y_pred_proba_train = model.predict_proba(X_train)
print(f"Accuracy (LR): {accuracy_score(y_train, y_pred_train):.4f}")
roc_auc_ovr = roc_auc_score(y_train, y_pred_proba_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (LR): {roc_auc_ovr:.4f}")


print("\n--- Step 8: Train and Evaluate Random Forest ---")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)
y_pred_rf_train = rf_model.predict(X_train)
y_pred_proba_rf_train = rf_model.predict_proba(X_train)
print(f"Accuracy (RF): {accuracy_score(y_train, y_pred_rf_train):.4f}")
roc_auc_rf_ovr = roc_auc_score(y_train, y_pred_proba_rf_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (RF): {roc_auc_rf_ovr:.4f}")


print("\n--- Step 9: Train and Evaluate Decision Tree ---")
dt_model = DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10)
dt_model.fit(X_train, y_train)
y_pred_dt_train = dt_model.predict(X_train)
y_pred_proba_dt_train = dt_model.predict_proba(X_train)
print(f"Accuracy (DT): {accuracy_score(y_train, y_pred_dt_train):.4f}")
roc_auc_dt_ovr = roc_auc_score(y_train, y_pred_proba_dt_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (DT): {roc_auc_ovr:.4f}")


print("\n--- Step 10: Prepare Data for Modeling (Post-Feature Engineering) ---")
non_feature_cols = ['ID', 'Customer_ID', 'SSN']
X_train_fe = df_train_fe.drop(columns=['Credit_Score'] + [col for col in non_feature_cols if col in df_train_fe.columns], errors='ignore')
y_train_fe = df_train_fe['Credit_Score']
X_test_fe = df_test_fe.drop(columns=[col for col in non_feature_cols if col in df_test_fe.columns], errors='ignore')

# Align columns for X_test_fe (post-FE)
train_fe_cols = set(X_train_fe.columns)
test_fe_cols = set(X_test_fe.columns)

missing_in_test_fe = list(train_fe_cols - test_fe_cols)
for col in missing_in_test_fe:
    X_test_fe[col] = 0

missing_in_train_fe = list(test_fe_cols - train_fe_cols)
for col in missing_in_train_fe:
    X_train_fe[col] = 0

X_test_fe = X_test_fe[X_train_fe.columns]
print("X_train_fe, y_train_fe, X_test_fe (post-FE) prepared.")


print("\n--- Step 11: Train and Evaluate Logistic Regression (with FE) ---")
lr_model_fe = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
lr_model_fe.fit(X_train_fe, y_train_fe)
y_pred_lr_fe_train = lr_model_fe.predict(X_train_fe)
y_pred_proba_lr_fe_train = lr_model_fe.predict_proba(X_train_fe)
print(f"Accuracy (LR with FE): {accuracy_score(y_train_fe, y_pred_lr_fe_train):.4f}")
roc_auc_lr_fe_ovr = roc_auc_score(y_train_fe, y_pred_proba_lr_fe_train, multi_class='ovr', average='macro')
print(f"ROC-AUC (LR with FE): {roc_auc_lr_fe_ovr:.4f}")

# Initialize Stratified K-Fold Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("\nStratifiedKFold (skf) initialized.")
print("All data preparation and initial model training/evaluation steps re-executed.")

# Define the models
models = {
    "Logistic Regression": LogisticRegression(solver='liblinear', random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10)
}

# Define scoring metrics
scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc_ovr']

results = {}

print("--- Starting Cross-Validation ---")

for name, model in models.items():
    print(f"\nEvaluating {name}...")
    cv_results = cross_validate(model, X_train_fe, y_train_fe, cv=skf, scoring=scoring, n_jobs=-1, verbose=0)

    results[name] = {
        'Accuracy': np.mean(cv_results['test_accuracy']),
        'Precision': np.mean(cv_results['test_precision_macro']),
        'Recall': np.mean(cv_results['test_recall_macro']),
        'F1-Score': np.mean(cv_results['test_f1_macro']),
        'ROC-AUC': np.mean(cv_results['test_roc_auc_ovr'])
    }

    print(f"{name} - Mean Accuracy: {results[name]['Accuracy']:.4f} (+/- {np.std(cv_results['test_accuracy']):.4f})")
    print(f"{name} - Mean Precision: {results[name]['Precision']:.4f} (+/- {np.std(cv_results['test_precision_macro']):.4f})")
    print(f"{name} - Mean Recall: {results[name]['Recall']:.4f} (+/- {np.std(cv_results['test_recall_macro']):.4f})")
    print(f"{name} - Mean F1-Score: {results[name]['F1-Score']:.4f} (+/- {np.std(cv_results['test_f1_macro']):.4f})")
    print(f"{name} - Mean ROC-AUC: {results[name]['ROC-AUC']:.4f} (+/- {np.std(cv_results['test_roc_auc_ovr']):.4f})")

print("\n--- Cross-Validation Complete ---")

# Display all results in a DataFrame for easier comparison
results_df = pd.DataFrame(results).T
print("\nSummary of Cross-Validation Results:")
display(results_df)

--- Step 1: Loading Datasets ---


/tmp/ipykernel_2075/1891814964.py:11: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('/content/train.csv.zip', compression='zip')


Train data loaded successfully.
Test data loaded successfully.

--- Step 2: Data Cleaning and Type Conversion ---
Data cleaning and conversion complete.

--- Step 3: Handling Missing Values ---
Missing value imputation complete.

--- Step 4: Categorical Feature Encoding ---


/tmp/ipykernel_2075/1891814964.py:109: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_train[col].fillna(median_val_train, inplace=True)
/tmp/ipykernel_2075/1891814964.py:112: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True

Categorical encoding complete.

--- Step 5: Feature Engineering ---


/tmp/ipykernel_2075/1891814964.py:157: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_copy[col].fillna(df_copy[col].median(), inplace=True)
/tmp/ipykernel_2075/1891814964.py:177: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=

Feature engineering complete. DataFrames: df_train_fe, df_test_fe created.

--- Step 6: Prepare Data for Modeling (Pre-Feature Engineering) ---
